# Notebook 02: Clustering with Mean Shift

---

## What This Notebook Covers

This notebook builds the **Mean Shift** clustering algorithm from scratch — first as a Python loop over individual points, then as a GPU-accelerated batched implementation. Along the way we develop the *exact* shape-manipulation patterns that show up in attention mechanisms, diffusion models, and any tensor-heavy code. We will learn:

1. **Generating synthetic clustered data** — multivariate-normal sampling, covariance matrices, building a 1,500-point dataset with 6 known cluster centers as ground truth
2. **The mean shift algorithm** — what it does, why it's clever, and what makes it different from K-Means (no need to specify *k*; clusters can be any shape)
3. **The Gaussian and triangular kernels** — converting distances to weights, and what the bandwidth parameter controls geometrically
4. **Broadcasting for pairwise distances** — the `a[None] - b[:, None]` pattern that lets us compute every point-to-point distance in one tensor expression
5. **Batched mean shift** — replacing a Python loop over points with a matmul, the same move that buys us 100×+ speedups in notebook 01
6. **Connecting mean shift to attention** — the surprising fact that the structure of the batched update is *literally* the structure of softmax attention
7. **GPU acceleration** — the one-line move that takes us from ~500 ms to ~2 ms for the full algorithm

---

## Why Mean Shift?

Mean shift is the *natural* clustering algorithm for someone trying to find dense regions in data with no labels and no prior on the cluster count. Unlike K-Means, which forces you to pick *k* up front and only works for blob-shaped clusters, mean shift just *finds* the modes of the density — however many there are, whatever shapes they have. As a teaching vehicle it's almost ideal:

- The algorithm itself is concrete enough to implement in 10 lines.
- The "why does it converge?" question has a beautiful answer (gradient ascent on a kernel density estimate) — deep-dived in §6.
- The vectorization story is the *same* story as matmul in notebook 01, but with one new wrinkle: the pairwise-distance computation needs an extra broadcasting axis.
- The final batched update step `(weight @ X) / weight.sum(...)` is *the same operation* as attention's value-aggregation step — deep-dived in §8.

So we get a useful clustering algorithm and a clean preview of attention from one notebook.

---

## Prerequisites

Comfortable with:

- **Notebook 01** — broadcasting rules, einsum decoding, the `@` operator. The pairwise-distance derivation here is exactly Example 4 of the broadcasting deep dive from 01.
- The idea of a **Gaussian distribution** in one dimension. The kernel and the multivariate-normal sampler both extend the 1-D bell curve to 2-D.
- Basic **PyTorch tensors** — slicing, `.shape`, `.sum(dim=...)`, `.clone()`.

If any of those are rusty, run through the relevant sections of `01_matmul_explained.ipynb` first.

---

## Relationship to `02_meanshift.ipynb`

This notebook builds on the historical-naming `02_meanshift.ipynb` (the user's annotated version using the early `_with_comments`-style naming). It adds:

1. The standard `_explained` top-of-file structure (what you're reading now).
2. **Three deep-dive sections:**
   - *Kernels and Bandwidth* — the math behind kernel choice and the geometry of bandwidth
   - *Why Mean Shift Converges* — the algorithm as gradient ascent on a KDE
   - *From Sequential to Batched (and Attention)* — the literal structural equivalence to attention
3. **"What does the code above do?" cells** after non-self-explanatory code, with explicit shape arithmetic and back-references to the broadcasting/einsum deep dives in notebook 01.

The historical `02_meanshift.ipynb` stays untouched per the 01–03 naming convention.

---

### What is Clustering?

**Clustering** is an **unsupervised learning** technique - meaning we don't have labels telling us which group each data point belongs to. Instead, the algorithm discovers groups (clusters) based on the structure of the data itself.

**Applications include:**
- Customer segmentation (grouping similar customers)
- Image segmentation (grouping similar pixels)
- Anomaly detection (finding points that don't fit any cluster)
- Document organization (grouping similar documents)

The easiest way to understand clustering is to see it in action with some generated data.

In [ ]:
# ============================================================================
# IMPORTING REQUIRED LIBRARIES
# ============================================================================
# In Python, we first need to import the tools (libraries) we'll use.
# Think of this like gathering your ingredients before cooking!

import math                    # Built-in Python math library
                               # Gives us constants like pi (3.14159...)
                               # and functions like sqrt (square root)

import matplotlib.pyplot as plt  # The standard plotting library in Python
                                  # 'plt' is the conventional short name
                                  # We'll use it to create visualizations

import operator                # Provides functional versions of operations
                               # Like operator.add for the + operation
                               # (We won't use this much in this notebook)

import torch                   # PyTorch - our main tensor computation library!
                               # This is the core library for deep learning
                               # It handles all our numerical computations
                               # and can run on GPU for huge speedups

from functools import partial  # A utility for creating modified functions
                               # partial(func, arg1) creates a new function
                               # where arg1 is already filled in
                               # Example: partial(add, 5) creates a function 
                               # that adds 5 to its input

# Quick check that PyTorch is installed correctly
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA (GPU) available: {torch.cuda.is_available()}")

**What does the code above do?**

Brings in five libraries the rest of the notebook will lean on:

| Module | Why we need it here |
|---|---|
| `math` | Stdlib — used for `sqrt(2π)` inside the Gaussian kernel normalizer. |
| `matplotlib.pyplot` | Plotting clusters, kernel curves, and the animated mean-shift run. |
| `operator` | Functional versions of the basic Python operators. Not heavily used in this notebook; appears for completeness. |
| `torch` | The data primitive. Every cluster point, every distance, every weight lives in a `torch.Tensor`. |
| `functools.partial` | Lets us pre-fill kernel arguments — e.g. `partial(gaussian, bw=2.5)` becomes a function of just the distance. We need this to pass parameterized kernels to `plot_func`. |

The two `print` lines at the bottom are a sanity check: which PyTorch version is loaded, and is a CUDA-capable GPU visible? The GPU check matters in §8 — without a GPU, the GPU-accelerated mean shift will silently run on CPU (still correct, just less dramatic speedup).

In [ ]:
# ============================================================================
# SETTING UP PYTORCH FOR REPRODUCIBILITY
# ============================================================================

# Set a random seed so we get the same "random" numbers every time
# This is crucial for reproducibility - same seed = same results
torch.manual_seed(42)

# Configure how tensors are displayed:
# - precision=3: Show 3 decimal places
# - linewidth=140: How wide the output can be before wrapping
# - sci_mode=False: Don't use scientific notation (e.g., show 0.001 not 1e-3)
torch.set_printoptions(precision=3, linewidth=140, sci_mode=False)

---
## Part 1: Creating Synthetic Clustered Data

### Why create synthetic data?

When learning algorithms, it's helpful to use data where we **know the answer**. By creating our own clustered data, we can:
1. Verify our algorithm finds the correct clusters
2. Visualize what's happening at each step
3. Control the difficulty (number of clusters, overlap, etc.)

### Our approach:
1. Choose random **centroids** (cluster centers)
2. Generate random points around each centroid
3. Combine all points into one dataset

In [ ]:
# ============================================================================
# DEFINING THE DATA PARAMETERS
# ============================================================================

# How many clusters do we want?
n_clusters = 6

# How many points per cluster?
n_samples = 250

# Total points will be: 6 clusters × 250 samples = 1,500 points
print(f"We will create {n_clusters} clusters with {n_samples} points each")
print(f"Total data points: {n_clusters * n_samples}")

### Creating Random Centroids

A **centroid** is the center point of a cluster. We'll randomly place 6 centroids in a 2D space ranging from -35 to +35 in both x and y directions.

In [ ]:
# ============================================================================
# GENERATING RANDOM CLUSTER CENTERS (CENTROIDS)
# ============================================================================

# torch.rand(n_clusters, 2) creates a tensor of shape (6, 2)
# - 6 rows (one for each cluster)
# - 2 columns (x and y coordinates)
# Values are between 0 and 1

# We multiply by 70 to get range [0, 70]
# Then subtract 35 to get range [-35, 35]
centroids = torch.rand(n_clusters, 2) * 70 - 35

print("Our 6 cluster centers (centroids):")
print(centroids)
print()
print("Each row is [x, y] coordinates of a cluster center")

### Generating Points Around Each Centroid

To create realistic clusters, we'll use a **multivariate normal distribution** (also called Gaussian distribution). This creates a "cloud" of points centered around each centroid.

**What is a multivariate normal distribution?**
- It's the 2D (or higher) version of the bell curve
- Points are most likely to be near the center (mean)
- Points further from the center are increasingly rare
- The **covariance matrix** controls the spread and shape

### Understanding Tensor Shapes - A Visual Guide

Before we continue, let's make sure we understand tensor shapes. This is **critical** for understanding the code!

**Shape notation**: When we say a tensor has shape `(3, 2)`, it means:
- **3 rows** (first number)
- **2 columns** (second number)

```
Shape (3, 2) looks like:    Shape (4,) looks like:
┌─────┬─────┐               ┌─────┬─────┬─────┬─────┐
│ 1.2 │ 3.4 │               │ 1.0 │ 2.0 │ 3.0 │ 4.0 │
├─────┼─────┤               └─────┴─────┴─────┴─────┘
│ 5.6 │ 7.8 │               (This is 1D - just a row)
├─────┼─────┤
│ 9.0 │ 1.1 │
└─────┴─────┘
```

**In this notebook:**
- Each **data point** is 2D (x, y coordinates)
- Data points are stored as **rows** in a tensor
- `shape (1500, 2)` = 1500 points, each with 2 coordinates

In [ ]:
# ============================================================================
# IMPORTING DISTRIBUTION TOOLS
# ============================================================================

# MultivariateNormal lets us sample from a multi-dimensional normal distribution
from torch.distributions.multivariate_normal import MultivariateNormal
from torch import tensor

# 'tensor' is a shortcut for creating PyTorch tensors
# Instead of torch.tensor([1,2,3]), we can write tensor([1,2,3])

### Understanding the Covariance Matrix

The **covariance matrix** controls the "shape" of our cluster:

```
Covariance [[5, 0],     Covariance [[10, 0],    Covariance [[5, 4],
            [0, 5]]                 [0, 2]]                 [4, 5]]
            
    ●●●●                  ●●●●●●●●                ●●●●
   ●●●●●●                  ●●●●●●                ●●●●●●
  ●●●●●●●●                 ●●●●●●               ●●●●●●●●
   ●●●●●●                  ●●●●●●                 ●●●●●●
    ●●●●                  ●●●●●●●●                  ●●●●
    
  (Circle)             (Wide ellipse)        (Tilted ellipse)
  Same spread          More x, less y        x and y correlated
  in x and y           spread                (diagonal tilt)
```

For this notebook, we use `[[5, 0], [0, 5]]` which creates **circular clusters** - equal spread in all directions, no correlation between x and y.

In [ ]:
# ============================================================================
# FUNCTION TO GENERATE POINTS AROUND A CENTER
# ============================================================================

def sample(m):
    """
    Generate n_samples random points centered around point m.

    Args:
        m: The center point (mean) - a tensor of shape (2,) for [x, y]

    Returns:
        A tensor of shape (n_samples, 2) containing random points

    How it works:
        1. MultivariateNormal creates a 2D normal distribution
        2. The mean is 'm' (our centroid)
        3. torch.diag(tensor([5., 5.])) creates the covariance matrix:
           [[5, 0],
            [0, 5]]
           This means spread of 5 in x direction and 5 in y direction,
           with no correlation between x and y (circular clusters)
        4. .sample((n_samples,)) draws n_samples random points
    """
    # Create a 2D normal distribution centered at m
    # torch.diag creates a diagonal matrix - controls spread in each direction
    covariance = torch.diag(tensor([5., 5.]))  # Spread of 5 in both directions
    distribution = MultivariateNormal(m, covariance)

    # Draw n_samples random points from this distribution
    return distribution.sample((n_samples,))

# Test with one centroid
test_samples = sample(centroids[0])
print(f"Generated {test_samples.shape[0]} points around centroid {centroids[0]}")
print(f"First 5 points:\n{test_samples[:5]}")

**What does the code above do?**

Defines `sample(m)` — a function that returns 250 random 2-D points drawn from a Gaussian centered at the input `m` with covariance `diag([5, 5])` (a circular blob of radius ~√5 ≈ 2.2).

The Python stdlib has `random.gauss` for scalar Gaussians, but for *multivariate* (correlated 2-D-and-up) Gaussians we use `torch.distributions.MultivariateNormal`. It takes a mean vector and a covariance matrix and exposes a `.sample()` method that draws from the resulting distribution. The covariance matrix controls cluster *shape*: `diag([5, 5])` is circular (no x/y correlation, same spread in both axes), `diag([20, 2])` would be a horizontal ellipse, and a non-diagonal matrix like `[[5, 4], [4, 5]]` would be a tilted ellipse.

The test call at the end shows that calling `sample(centroids[0])` returns 250 points clustered around the first centroid — exactly what we'll repeat for the other 5 centroids in the next cell.

In [ ]:
# ============================================================================
# GENERATING ALL DATA POINTS
# ============================================================================

# Generate n_samples points around each centroid
# This creates a list of 6 tensors, each of shape (250, 2)
slices = [sample(c) for c in centroids]

print(f"Number of slices: {len(slices)}")
print(f"Shape of each slice: {slices[0].shape}")

# Combine all slices into one big tensor
# torch.cat concatenates tensors along the first dimension (rows)
data = torch.cat(slices)

print(f"\nCombined data shape: {data.shape}")
print("This means we have 1500 points, each with 2 coordinates (x, y)")

### Visualizing Our Data

Let's create a function to visualize our clusters. We'll:
1. Plot each cluster in a different color
2. Mark the true centroids with X markers

This visualization will help us verify that our clustering algorithm works correctly.

In [ ]:
# ============================================================================
# FUNCTION TO VISUALIZE CLUSTERED DATA
# ============================================================================

def plot_data(centroids, data, n_samples, ax=None):
    """
    Plot clustered data with centroids marked.

    Args:
        centroids: Tensor of shape (n_clusters, 2) - cluster centers
        data: Tensor of shape (n_total, 2) - all data points
        n_samples: Number of samples per cluster (to separate colors)
        ax: Optional matplotlib axis to plot on

    How the coloring works:
        - Points 0 to 249 belong to cluster 0
        - Points 250 to 499 belong to cluster 1
        - And so on...
        - We use this knowledge to color each cluster differently
    """
    # Create a new figure if no axis provided
    if ax is None:
        _, ax = plt.subplots()

    # Loop through each cluster
    for i, centroid in enumerate(centroids):
        # Extract points belonging to this cluster
        # Cluster i has points from index i*n_samples to (i+1)*n_samples
        start_idx = i * n_samples
        end_idx = (i + 1) * n_samples
        samples = data[start_idx:end_idx]

        # Plot the data points (s=1 makes them small dots)
        ax.scatter(samples[:, 0], samples[:, 1], s=1)

        # Plot the centroid with an X marker
        # *centroid unpacks [x, y] into separate arguments
        # First X is larger and black (background)
        ax.plot(*centroid, markersize=10, marker="x", color='k', mew=5)
        # Second X is smaller and magenta (foreground)
        ax.plot(*centroid, markersize=5, marker="x", color='m', mew=2)

**What does the code above do?**

Defines `plot_data` — a helper that visualizes the clustered data with each cluster in its own color and the true centroids marked with black-and-magenta X's.

The clever bit is the loop over clusters: since the data was concatenated cluster-by-cluster (250 points per cluster, in order), points 0–249 belong to cluster 0, 250–499 to cluster 1, etc. The slice `data[i*n_samples : (i+1)*n_samples]` extracts cluster `i` — no labels needed, just arithmetic on indices. This works because we *know* the underlying structure of the synthetic data. On real data without labels we'd just scatter all points in one color and let the clustering algorithm do its thing.

The double-X plotting trick (one large black, one small magenta) is a common matplotlib idiom for making markers visible regardless of background color — the black X provides contrast against any light background; the magenta provides contrast against any dark background.

In [ ]:
# ============================================================================
# VISUALIZE OUR GENERATED DATA
# ============================================================================

# Plot our synthetic clustered data
plot_data(centroids, data, n_samples)
plt.title("Our Synthetic Clustered Data\n(X marks show true cluster centers)")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")
plt.show()

print("Each color represents points from one cluster.")
print("The X markers show the true centers we used to generate the data.")
print("Our goal: Can the Mean Shift algorithm find these centers?")

---
## Part 2: The Mean Shift Algorithm

### What is Mean Shift?

**Mean Shift** is a clustering algorithm that finds dense regions in data by iteratively moving points toward areas of higher density. Think of it like balls rolling downhill - they naturally collect at the bottom (the dense regions).

### Why Mean Shift instead of K-Means?

Most people learn **K-Means** first, but Mean Shift has important advantages:

| Feature | K-Means | Mean Shift |
|---------|---------|------------|
| Number of clusters | Must specify in advance | Automatically determined |
| Cluster shape | Must be roughly spherical | Can be any shape |
| Sensitivity to outliers | High | Lower |
| Computational cost | Lower | Higher |

### Connection to Diffusion Models

The Mean Shift algorithm is conceptually similar to the **denoising process** in Stable Diffusion:

- **Mean Shift**: Points move toward high-density regions (cluster centers)
- **Diffusion Denoising**: Noisy latent representations move toward the "manifold" of real images

Both involve iteratively updating positions based on nearby points!

### Worked Example: Mean Shift on 4 Points

Let's trace through one iteration of Mean Shift by hand with just 4 points:

```
Points: A=(0,0), B=(1,0), C=(10,0), D=(11,0)

Step 1: Focus on point A=(0,0)

Step 2: Calculate distances from A to all points:
        d(A,A) = 0,  d(A,B) = 1,  d(A,C) = 10,  d(A,D) = 11

Step 3: Apply Gaussian kernel (bandwidth=2) to get weights:
        w_A = 0.199,  w_B = 0.176,  w_C ≈ 0,  w_D ≈ 0
        (Notice: C and D are so far that their weights are essentially 0!)

Step 4: Compute weighted average:
        new_A = (0.199×A + 0.176×B + 0×C + 0×D) / (0.199 + 0.176 + 0 + 0)
              = (0.199×[0,0] + 0.176×[1,0]) / 0.375
              = [0,0] + [0.176,0]) / 0.375
              = [0.47, 0]
        
        Point A moves from (0,0) to (0.47, 0) - toward point B!
```

**After many iterations:**
- Points A and B converge to the same location (cluster 1)
- Points C and D converge to the same location (cluster 2)
- The algorithm automatically found 2 clusters!

---

### Understanding Step 3: How the Gaussian Kernel Works

**What is the Gaussian Kernel?**

The Gaussian kernel is a mathematical function that converts **distances** into **weights**:
- **Close points** (small distance) → **High weight** (strong influence)
- **Far points** (large distance) → **Low weight** (weak/no influence)

**Simple Analogy:**

Imagine you're in a crowd trying to find the "center" of people:
- **Without kernel**: Everyone counts equally - even someone 100 meters away
- **With Gaussian kernel**: Nearby people have a LOUD voice, distant people are barely heard

**The Formula:**

```
weight = exp(-0.5 × (distance/bandwidth)²) / (bandwidth × √(2π))
```

**Why do C and D have weight ≈ 0?**

The Gaussian kernel drops off **very quickly** as distance increases:

| Distance | Weight (bw=2) | Percentage of Max |
|----------|---------------|-------------------|
| 0 | 0.199 | 100% |
| 1 | 0.176 | 88% |
| 2 | 0.121 | 61% |
| 3 | 0.065 | 33% |
| 4 | 0.027 | 14% |
| 5 | 0.009 | 4% |
| 10 | ≈ 0.000 | 0.0001% |

Points C (distance=10) and D (distance=11) are so far that their weights are essentially **zero**. This is exactly what we want - the algorithm only cares about **nearby** points!

**The Bandwidth Parameter:**

- **Small bandwidth (bw=1)**: Only very close points matter → Many small clusters
- **Large bandwidth (bw=5)**: Distant points also matter → Fewer large clusters
- **bw=2** (our example): A good balance

**Visual Representation:**

```
Weight
  │
  │  ██           ← Point A (distance=0, weight=0.199)
  │ ████          ← Point B (distance=1, weight=0.176)
  │██████
  │████████
  │██████████
  └─────────────────────────── Distance
     0  1  2  3  4  5 ... 10  11
     A  B              C   D (weights ≈ 0)
```

In [ ]:
# ============================================================================
# INTERACTIVE DEMONSTRATION: Mean Shift on 4 Points
# ============================================================================
# Let's verify the worked example above with actual code!
# NOTE: We'll define the gaussian function first, then demonstrate

# First, let's define the gaussian function we'll use
def gaussian(d, bw):
    """Gaussian kernel - converts distance to weight."""
    return torch.exp(-0.5 * (d/bw)**2) / (bw * math.sqrt(2*math.pi))

# Create our 4 simple points
demo_points = torch.tensor([
    [0., 0.],   # Point A
    [1., 0.],   # Point B  
    [10., 0.],  # Point C
    [11., 0.]   # Point D
])

print("Our 4 demo points:")
print("A = (0, 0)")
print("B = (1, 0)")
print("C = (10, 0)")
print("D = (11, 0)")
print()

# Focus on point A (index 0)
point_A = demo_points[0]

# Step 2: Calculate distances from A to all points
distances_from_A = torch.sqrt(((point_A - demo_points)**2).sum(1))
print("Step 2 - Distances from A to each point:")
for i, (name, dist) in enumerate(zip(['A', 'B', 'C', 'D'], distances_from_A)):
    print(f"  d(A, {name}) = {dist:.1f}")

# Step 3: Apply Gaussian kernel with bandwidth=2
bw = 2
weights = gaussian(distances_from_A, bw)
print(f"\nStep 3 - Weights (using Gaussian kernel, bandwidth={bw}):")
for name, w in zip(['A', 'B', 'C', 'D'], weights):
    print(f"  w_{name} = {w:.6f}")
print("Notice: Points C and D have tiny weights because they're far from A!")

# Step 4: Compute weighted average
numerator = (weights[:, None] * demo_points).sum(0)
denominator = weights.sum()
new_A = numerator / denominator
print(f"\nStep 4 - Weighted average calculation:")
print(f"  Numerator (sum of weight × point): {numerator}")
print(f"  Denominator (sum of weights): {denominator:.4f}")
print(f"  New position for A: {new_A}")
print(f"\nPoint A moved from (0, 0) to ({new_A[0]:.2f}, {new_A[1]:.2f})!")

**What does the code above do?**

Numerically reproduces the worked example from the markdown above on four points (A, B, C, D arranged on a line).

The step-by-step output confirms three things:

1. **Distances to C and D are large** (10 and 11 respectively), so the Gaussian kernel at `bw=2` gives them essentially zero weight. They effectively don't exist as far as point A is concerned.
2. **Distance to B is small** (1.0), so it dominates the weighted average.
3. **The new position for A** is around (0.47, 0) — pulled toward B but not all the way to it (the contribution of A itself, distance 0, still has some pull).

After one update step, A has moved 47% of the way toward B. After several iterations, A and B will converge to the same point — the midpoint of the {A, B} subcluster. C and D will similarly converge with each other. The algorithm has *automatically discovered two clusters* without being told how many to look for. That's the punchline of mean shift.

### The Mean Shift Algorithm Steps

For each data point x:

1. **Calculate distances**: Find how far x is from every other point
2. **Apply kernel weighting**: Points closer to x get higher weights (using Gaussian kernel)
3. **Compute weighted average**: Move x toward the weighted center of all points
4. **Repeat**: Do this for all points, then iterate multiple times

The **bandwidth** parameter controls how far we look for neighbors:
- Small bandwidth → Only very close points matter → Many small clusters
- Large bandwidth → Distant points also matter → Fewer large clusters

### 🎬 Interactive: watch the four steps run

The animation below executes the **exact four steps** defined above on a small 2-D point cloud. 
Use **Track one point** to follow a single point through *distance → weight → weighted average → shift* 
with the live math, or **All points at once** to watch every point converge into clusters. 
Step through manually (**Next step**), jump to any step, or auto-play.


In [ ]:
# ============================================================================
# INTERACTIVE: Mean Shift mechanics  (distance -> weight -> weighted avg -> shift)
# ============================================================================
# The full animation defined above, loaded from the published page and shown in
# an isolated <iframe> (its CSS/JS can't leak in). Works in Jupyter and Colab.
from IPython.display import IFrame

IFrame("https://shammun.github.io/shammunul-fastai-notes/notebooks/meanshift_mechanics.html",
       width="100%", height=960)


In [ ]:
# ============================================================================
# COMPUTING THE MEAN OF ALL DATA (FOR COMPARISON)
# ============================================================================

# Let's first see what happens if we just take the mean of all data
# data.mean(0) computes the mean along dimension 0 (across all points)
# This gives us the overall center of all the data
midp = data.mean(0)

print(f"Center of all data (midpoint): {midp}")
print()
print("This is NOT useful for clustering - it's just the overall center.")
print("Mean Shift will find multiple centers - one for each cluster!")

### The Gaussian Kernel - Heart of Mean Shift

The **Gaussian kernel** (also called RBF kernel) is a function that converts distances into weights:
- **Close points** (small distance) → **High weight** (contribute more)
- **Far points** (large distance) → **Low weight** (contribute less)

### Intuition: The "Voting" Analogy

Imagine you're at a party and want to find the "center" of where people are standing:

1. **Without kernel**: Every person gets 1 vote, regardless of distance. People far away have equal say as people nearby.

2. **With Gaussian kernel**: People close to you get a LOUD voice (high weight), people far away are barely heard (low weight). The "center" shifts toward where most nearby people are standing.

### The Formula (Don't Panic!)

$$K(d) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{1}{2}\left(\frac{d}{\sigma}\right)^2\right)$$

**Let's break this down piece by piece:**

| Part of Formula | What it does | Example |
|-----------------|--------------|---------|
| $d$ | The distance between two points | d = 3.5 units |
| $\sigma$ (sigma) | The **bandwidth** - controls how quickly weights decrease | σ = 2.5 |
| $(d/\sigma)^2$ | Normalized squared distance - scales by bandwidth | (3.5/2.5)² = 1.96 |
| $-\frac{1}{2}(...)$ | Negative half - makes the curve peak at d=0 | -0.5 × 1.96 = -0.98 |
| $\exp(...)$ | Exponential function (e^x) - creates the bell curve shape | e^(-0.98) ≈ 0.375 |
| $\frac{1}{\sigma\sqrt{2\pi}}$ | Normalization - makes the total area equal to 1 | 1/(2.5 × 2.507) ≈ 0.16 |

**Final result**: 0.16 × 0.375 ≈ 0.06 (the weight for distance 3.5 with bandwidth 2.5)

### Why is this important for diffusion models?

The Gaussian kernel appears everywhere in machine learning:
- **Attention mechanisms** in transformers use similar weighting
- **Diffusion models** add Gaussian noise during the forward process
- **RBF networks** use Gaussian basis functions
- **Gaussian blur** in image processing uses this same kernel!

In [ ]:
# ============================================================================
# IMPLEMENTING THE GAUSSIAN KERNEL
# ============================================================================

def gaussian(d, bw):
    """
    Compute Gaussian kernel weights from distances.

    Args:
        d: Distance(s) - can be a single number or a tensor of distances
        bw: Bandwidth (standard deviation) - controls the "reach"

    Returns:
        Weight(s) between 0 and ~0.4/bw (highest when d=0)

    The formula breaks down as:
        1. (d/bw)**2: Square of normalized distance
        2. -0.5 * ...: Negative half (for the Gaussian shape)
        3. torch.exp(...): e raised to this power
        4. / (bw * math.sqrt(2*math.pi)): Normalization factor

    Example:
        - d=0, bw=1: Returns ~0.399 (maximum value)
        - d=1, bw=1: Returns ~0.242 (about 60% of max)
        - d=2, bw=1: Returns ~0.054 (about 13% of max)
        - d=3, bw=1: Returns ~0.004 (nearly zero)
    """
    # Compute the Gaussian kernel
    exponent = -0.5 * ((d / bw) ** 2)  # -(d/bw)^2 / 2
    normalization = bw * math.sqrt(2 * math.pi)  # sqrt(2*pi) * bw
    return torch.exp(exponent) / normalization

# Test the function
print("Gaussian kernel values at different distances (bandwidth=1):")
for dist in [0, 0.5, 1, 2, 3]:
    weight = gaussian(torch.tensor(dist), bw=1)
    print(f"  Distance {dist}: Weight = {weight:.4f}")

**What does the code above do?**

Implements the Gaussian kernel as a vectorized function over distance tensors:

$$K(d) = \frac{1}{\sigma\sqrt{2\pi}}\,\exp\!\left(-\frac{1}{2}\left(\frac{d}{\sigma}\right)^2\right)$$

Two things to notice:

- **Single point or whole tensor — same code.** Because `torch.exp` and the arithmetic operators are elementwise, calling `gaussian(distances, bw)` where `distances` is a shape-(1500,) tensor gives back a shape-(1500,) tensor of weights, with no loop needed. This is the broadcasting pattern from notebook 01 in action: the scalar bandwidth `bw` broadcasts against every element.
- **The normalization factor `1/(σ·√(2π))` is what makes this a proper density.** It ensures the kernel integrates to 1 over all real distances. For mean shift specifically we don't actually need normalization — we'd divide by the sum of weights anyway — but it's there because the same formula is the textbook 1-D Gaussian PDF, and it's nice to keep that connection.

The test loop at the bottom shows the classic Gaussian decay shape: weight drops from ~0.4 at distance 0 to ~0.004 at distance 3 (with bw=1). Past distance ~3σ, weights are vanishingly small — the kernel has *effective* finite support even though it's technically defined on all of ℝ.

In [ ]:
# ============================================================================
# HELPER FUNCTION TO PLOT ANY FUNCTION
# ============================================================================

def plot_func(f):
    """
    Plot a function f over the range [0, 10].

    Args:
        f: A function that takes a tensor and returns a tensor

    This is useful for visualizing kernel functions.
    """
    # Create 100 evenly spaced points from 0 to 10
    x = torch.linspace(0, 10, 100)

    # Apply the function and plot
    plt.plot(x, f(x))
    plt.xlabel("Distance")
    plt.ylabel("Weight")

In [ ]:
# ============================================================================
# VISUALIZING THE GAUSSIAN KERNEL
# ============================================================================

# partial(gaussian, bw=2.5) creates a function that calls gaussian with bw=2.5
# So we can call it with just the distance: partial_gaussian(d) = gaussian(d, 2.5)
plot_func(partial(gaussian, bw=2.5))
plt.title("Gaussian Kernel (bandwidth=2.5)")
plt.xlabel("Distance from point")
plt.ylabel("Weight")
plt.show()

print("Notice:")
print("- Weight is highest when distance is 0 (the point itself)")
print("- Weight decreases smoothly as distance increases")
print("- By distance ~7, weight is essentially zero")

**What does the code above do?**

Plots the Gaussian kernel function over the distance range [0, 10] with bandwidth 2.5. `partial(gaussian, bw=2.5)` produces a function of one argument (just the distance) by freezing `bw=2.5`, which `plot_func` then evaluates and plots.

**What you should see:** a smooth bell-shaped curve that peaks at distance 0 (height ~0.16) and decays toward 0 as distance grows. By distance ~7 the curve is visually indistinguishable from the x-axis — that's about 3 bandwidths (3σ), which is where the Gaussian effectively dies out.

In [ ]:
# ============================================================================
# UNDERSTANDING PARTIAL FUNCTIONS
# ============================================================================

# 'partial' is a powerful tool from functools
# It creates a new function with some arguments pre-filled

print("What is 'partial'?")
print(f"partial is: {partial}")
print()

# Example: Create a function that always uses bw=2.5
gaussian_bw25 = partial(gaussian, bw=2.5)

# Now we can call it with just the distance
dist = torch.tensor(1.0)
print(f"gaussian(1.0, bw=2.5) = {gaussian(dist, bw=2.5):.4f}")
print(f"gaussian_bw25(1.0) = {gaussian_bw25(dist):.4f}")
print("Same result! partial just pre-fills the bw argument.")

In [ ]:
plot_func(lambda x: gaussian(x, bw=2.5))

**What does the code above do?**

Same kernel plot as above, but using a `lambda` instead of `partial`. Worth seeing both because they show up in different codebases — `partial` is the functional style (named function, pre-filled argument); `lambda` is the inline anonymous-function style.

Both produce identical output. Choose whichever reads better in context.

### Choosing the Bandwidth

The **bandwidth** is the most important parameter in Mean Shift:

- **Too small**: Each point becomes its own cluster
- **Too large**: Everything merges into one cluster
- **Just right**: Natural clusters are discovered

**Rule of thumb**: Choose bandwidth so that about 1/3 of the data falls within the bandwidth of a typical point.

In [ ]:
# ============================================================================
# TRIANGULAR KERNEL FUNCTION
# ============================================================================
#
# The triangular kernel is a simple alternative to the Gaussian kernel.
# It converts distances into weights, where:
#   - Close points (small distance) get HIGH weight
#   - Far points (large distance) get LOW weight
#   - Points beyond the "reach" get ZERO weight (hard cutoff)

def tri(d, i):
    """
    Triangular kernel - converts distance to weight with a hard cutoff.

    Args:
        d: Distance value(s) - can be a single number or tensor of distances
        i: The "reach" or cutoff distance - beyond this, weight is 0

    Returns:
        Weight(s) between 0 and 1:
          - At d=0: returns 1.0 (maximum weight)
          - At d=i: returns 0.0 (cutoff point)
          - At d>i: returns 0.0 (beyond cutoff)

    The formula: weight = max(0, -d + i) / i

    Step-by-step breakdown:
        1. (-d + i)      : Subtract distance from reach
                           At d=0: gives i (positive)
                           At d=i: gives 0
                           At d>i: gives negative number

        2. .clamp_min(0) : Replace any negative values with 0
                           This creates the "hard cutoff" - anything beyond
                           reach i becomes exactly 0

        3. / i           : Divide by i to normalize to range [0, 1]
                           At d=0: i/i = 1.0 (maximum)
                           At d=i: 0/i = 0.0 (cutoff)

    Example with i=8 (reach of 8 units):
        tri(0, 8) = (-0 + 8).clamp_min(0) / 8 = 8/8 = 1.0
        tri(2, 8) = (-2 + 8).clamp_min(0) / 8 = 6/8 = 0.75
        tri(4, 8) = (-4 + 8).clamp_min(0) / 8 = 4/8 = 0.5
        tri(6, 8) = (-6 + 8).clamp_min(0) / 8 = 2/8 = 0.25
        tri(8, 8) = (-8 + 8).clamp_min(0) / 8 = 0/8 = 0.0
        tri(10, 8) = (-10 + 8).clamp_min(0) / 8 = max(0,-2)/8 = 0/8 = 0.0

    Visual representation (i=8):

        Weight
        1.0 |*
            |  *
        0.5 |    *
            |      *
        0.0 |________*________ Distance
            0  2  4  6  8  10
                      ↑
                   cutoff (weight becomes 0)

    Comparison to Gaussian kernel:
        - Triangular: Hard cutoff at distance i (weight = exactly 0)
        - Gaussian: Soft decay (weight approaches 0 but never exactly 0)
        - Triangular is simpler and faster to compute
        - Gaussian is smoother and often gives better results
    """
    # Compute the triangular kernel:
    # Step 1: (-d + i) calculates how far we are from the cutoff
    # Step 2: .clamp_min(0) ensures no negative weights (hard cutoff)
    # Step 3: /i normalizes the result to range [0, 1]
    return (-d + i).clamp_min(0) / i

**What does the code above do?**

Defines the **triangular kernel** as a contrast to the Gaussian. The triangular kernel is dead simple:

- Weight = 1 at distance 0
- Weight decreases linearly with distance
- Weight = 0 at distance ≥ `i` (the "reach")

The implementation `(-d + i).clamp_min(0) / i` does the math in three steps:

1. `-d + i` is positive when d < i (we're inside the kernel's support) and negative when d > i (outside).
2. `.clamp_min(0)` replaces negatives with 0 — the hard cutoff.
3. `/ i` scales to a max of 1 at d=0.

The point of having both kernels available is pedagogical: it shows that mean shift doesn't care *which* kernel you use, only that it returns "small for far points, large for close points." The triangular kernel is faster to compute (no exponential), but the Gaussian's smoothness gives slightly better convergence in practice.

In [ ]:
# ============================================================================
# TESTING THE TRIANGULAR KERNEL
# ============================================================================

# Let's see how the triangular kernel assigns weights at different distances
# We're using reach=8, meaning:
#   - At distance 0: weight = 1.0 (maximum)
#   - At distance 8: weight = 0.0 (cutoff point)
#   - Beyond distance 8: weight = 0.0

print("Triangular kernel values (reach=8):")

# Test distances from 0 to 10
# Notice: 8 and 10 are at or beyond the reach, so weight becomes 0
for dist in [0, 2, 4, 6, 8, 10]:
    # Convert distance to a tensor (required by our tri function)
    # float(dist) ensures we get a floating-point tensor
    weight = tri(torch.tensor(float(dist)), i=8)
    
    # Print the result with 3 decimal places
    # f-string formatting: {weight:.3f} means "format weight as float with 3 decimals"
    print(f"  Distance {dist}: Weight = {weight:.3f}")

# Expected output:
#   Distance 0: Weight = 1.000  (at center, maximum weight)
#   Distance 2: Weight = 0.750  (25% of way to cutoff)
#   Distance 4: Weight = 0.500  (halfway to cutoff)
#   Distance 6: Weight = 0.250  (75% of way to cutoff)
#   Distance 8: Weight = 0.000  (at cutoff, weight becomes 0)
#   Distance 10: Weight = 0.000 (beyond cutoff, stays 0)

# ============================================================================
# UNDERSTANDING: weight = tri(torch.tensor(float(dist)), i=8)
# ============================================================================
#
# Let's break down this line step by step (using dist=4 as example):
#
# Step 1: dist = 4 (an integer from the loop)
#
# Step 2: float(dist) = 4.0
#         Convert integer to floating-point number
#
# Step 3: torch.tensor(float(dist)) = tensor(4.0)
#         Create a PyTorch tensor from the float
#         (Required because tri() uses PyTorch operations like .clamp_min())
#
# Step 4: tri(tensor(4.0), i=8)
#         Call the triangular kernel function with:
#           - d = tensor(4.0)  (the distance)
#           - i = 8            (the reach/cutoff)
#
#         Inside tri(d, i):
#           return (-d + i).clamp_min(0) / i
#           = (-4 + 8).clamp_min(0) / 8
#           = (4).clamp_min(0) / 8
#           = 4 / 8
#           = 0.5
#
# Step 5: weight = 0.5
#         Store the result
#
# Visual summary:
#   dist=4 -> float(4) -> torch.tensor(4.0) -> tri(..., i=8) -> 0.5
#              |              |                      |
#            "4.0"      tensor(4.0)         Weight = 0.5 (halfway to cutoff)

In [ ]:
# ============================================================================
# VISUALIZING THE TRIANGULAR KERNEL
# ============================================================================

plot_func(partial(tri, i=8))
plt.title("Triangular Kernel (reach=8)")
plt.xlabel("Distance from point")
plt.ylabel("Weight")
plt.show()

print("The triangular kernel:")
print("- Is 1 at distance 0")
print("- Decreases linearly")
print("- Becomes exactly 0 at distance 8 (the reach)")
print("- Simpler than Gaussian but less smooth")

**What does the code above do?**

Plots the triangular kernel. Compare to the Gaussian plot two cells above:

| Property | Gaussian | Triangular |
|---|---|---|
| Maximum value | ~0.16 (with bw=2.5) | 1.0 (with reach=8) |
| Shape | Smooth bell | Sharp wedge |
| Where does weight reach zero? | Asymptotically (gets close as $d \to \infty$) | Exactly at $d = i$ (reach) |
| Differentiable? | Yes, everywhere | No (kink at d=0, drop to 0 at d=i) |

The triangular kernel's hard cutoff means points beyond reach `i` are *exactly* ignored — no floating-point fuzz, no influence at all. The Gaussian's soft cutoff means even very distant points have tiny non-zero weight, which is more "honest" probabilistically but slightly slower to converge.

The choice between them is largely aesthetic / pedagogical for mean shift. Both work.

---

# Deep Dive: Kernels and Bandwidth

We've used both the Gaussian and triangular kernels above without saying *why* this whole "convert distance to weight" idea is the right thing to do for clustering. This section answers that — and explains what the bandwidth parameter actually controls geometrically.

---

## What is a Kernel?

In the kernel-methods literature, a **kernel** is a function $K(x, y)$ that measures similarity between two points. The "kernel" we're using here is technically the **radial basis kernel** (depends only on distance, not the points themselves):

$$K(x, y) = K_\sigma(\|x - y\|)$$

where $K_\sigma$ is a univariate function parameterized by a bandwidth $\sigma$. Common choices:

| Kernel | Formula $K_\sigma(d)$ | Support |
|---|---|---|
| **Gaussian** (RBF) | $\frac{1}{\sigma\sqrt{2\pi}}\exp\!\left(-\frac{d^2}{2\sigma^2}\right)$ | $(-\infty, \infty)$ (effective: $\sim 3\sigma$) |
| **Triangular** | $\frac{1}{\sigma}\max\!\left(0,\ 1 - \frac{d}{\sigma}\right)$ | $[-\sigma, \sigma]$ (hard) |
| **Epanechnikov** | $\frac{3}{4\sigma}\max\!\left(0,\ 1 - \frac{d^2}{\sigma^2}\right)$ | $[-\sigma, \sigma]$ (hard) |
| **Uniform / "box"** | $\frac{1}{2\sigma}\,\mathbb{1}_{d \le \sigma}$ | $[-\sigma, \sigma]$ (hard) |

Any of these would work for mean shift. The Gaussian is just the most common default because:

1. **It's smooth everywhere** — the resulting density estimate is differentiable, which makes gradient-based arguments (like the convergence proof in the next deep dive) clean.
2. **It has unbounded support** — every point has nonzero influence on every other point. In practice the influence dies off so fast that this doesn't matter (3σ rule), but theoretically it removes some edge effects you'd see with hard-cutoff kernels.
3. **It's the maximum-entropy kernel** with given variance — the "least committal" choice.

The triangular and Epanechnikov kernels are slightly faster (no exponential) and often used in production. The uniform kernel is rarely a good choice — it makes the density estimate discontinuous.

---

## Why Squared Distance?

You might wonder: why is the input to the kernel $d^2$ (in the Gaussian's exponent) rather than $|d|$ or $d^4$ or something else? The mathematical reason is **isotropy**: in $d$ dimensions, the only smooth radially-symmetric Gaussian function is $\exp(-\|x\|^2 / 2\sigma^2)$. Linear or higher-order exponents wouldn't have this property — they'd produce weights that distinguish between "two units northeast" and "two units east" in subtle ways.

For the triangular kernel, we use $|d|$ (not $d^2$) precisely because the Gaussian-isotropy argument doesn't apply — the triangular kernel is *not* smooth at the origin anyway, so we can keep the math simple.

In every case, **the kernel depends only on distance** — direction is irrelevant. That's the "radial" in "radial basis function."

---

## What the Bandwidth Actually Controls

The bandwidth $\sigma$ controls the kernel's *scale* — roughly, "how big is the neighborhood that each point cares about?"

For the Gaussian kernel specifically, by the time you're $3\sigma$ away from the center, the weight has dropped to about $\exp(-4.5) \approx 0.011$ of its peak — essentially zero. So $\sigma$ acts as a *soft* neighborhood radius:

- $\sigma = 1$: a point's "neighborhood" extends about 3 units.
- $\sigma = 2.5$ (our default): neighborhood extends about 7.5 units.
- $\sigma = 10$: neighborhood extends about 30 units.

For mean shift specifically, the bandwidth is the algorithm's *only* knob. It controls everything that matters:

| $\sigma$ regime | What happens |
|---|---|
| Too small (say σ < typical within-cluster spread) | Each point sees only its immediate neighbors. Clusters break into many small subclusters. Algorithm converges, but to dozens of "modes" instead of the true cluster count. |
| Too large (σ ≫ between-cluster distance) | Every point sees every other point with roughly equal weight. All points converge to the global centroid. One giant cluster. |
| Goldilocks (σ ≈ within-cluster spread, much less than between-cluster distance) | Each point converges to its true cluster's mode. The algorithm "discovers" the right number of clusters automatically. |

The rule of thumb: pick $\sigma$ so that the kernel covers about 1/3 of the data points around a typical point. For our 1500-point dataset with 6 clusters of spread ~2.2, that puts $\sigma$ somewhere in the 2–3 range. Bandwidth selection is a real subfield of statistics; for serious work, "Silverman's rule" or cross-validation are standard data-driven choices.

---

## Connection to Kernel Density Estimation (KDE)

What we're really doing with these kernel weights is implicitly building a **kernel density estimate** of the data:

$$\hat{p}(x) = \frac{1}{n}\sum_{i=1}^{n} K_\sigma(x - x_i)$$

That's it. Place a tiny Gaussian (or triangle) bump on each data point, then add them up to get a smooth density. The result $\hat{p}(x)$ is high in regions where data points cluster and low in empty regions.

The bandwidth $\sigma$ controls the *smoothness* of this density estimate:

- Small $\sigma$: density has 1500 sharp spikes (one per point), looks noisy.
- Large $\sigma$: density is one smooth blob covering all points, hides cluster structure.
- Right $\sigma$: density has clear peaks at the cluster centers and valleys in between.

The next deep dive will show that mean shift is *literally* gradient ascent on this density estimate. Bandwidth selection for mean shift and bandwidth selection for KDE are the same problem in two different costumes.

### What is Broadcasting? A Visual Explanation

**Broadcasting** is PyTorch's clever way of performing operations on tensors of different shapes. It "stretches" smaller tensors to match larger ones.

**Example: Adding a number to every element**
```
[1, 2, 3] + 10 = [11, 12, 13]
```
The `10` is "broadcast" (copied) to match `[10, 10, 10]`.

**Example: Subtracting vectors**
```
Point x = [5, 3]

All points X:                 x - X (how it works):
┌─────┬─────┐                 x is "broadcast" to every row
│ 2.0 │ 1.0 │  →  [5,3] - [2,1] = [3, 2]
├─────┼─────┤
│ 4.0 │ 2.0 │  →  [5,3] - [4,2] = [1, 1]
├─────┼─────┤
│ 6.0 │ 4.0 │  →  [5,3] - [6,4] = [-1, -1]
└─────┴─────┘
```

**The Rule**: Dimensions are compatible when:
1. They are equal, OR
2. One of them is 1 (it gets stretched)

```
(2,)    and (3, 2)  → Compatible! (2,) becomes (3, 2)
(5, 1)  and (1, 3)  → Compatible! Result is (5, 3)
(5, 2)  and (3, 2)  → NOT compatible (5 ≠ 3)
```

### Computing Distances in Mean Shift

To apply the kernel, we first need to compute distances. Let's walk through this step by step using **broadcasting** - a powerful PyTorch feature.

In [ ]:
# ============================================================================
# SETTING UP FOR DISTANCE CALCULATION
# ============================================================================

# Make a copy of our data (we'll modify it during mean shift)
X = data.clone()

# Let's focus on the first point
x = data[0]

print(f"First data point x: {x}")
print(f"Shape of x: {x.shape}")
print(f"Shape of all data X: {X.shape}")

In [ ]:
# ============================================================================
# UNDERSTANDING BROADCASTING FOR SUBTRACTION
# ============================================================================

# We want to compute (x - X) for all points in X
# x has shape (2,) and X has shape (1500, 2)
# How can we subtract a vector from a matrix?

# Method 1: Add a dimension to x using x[None]
# x[None] has shape (1, 2) - adds a new dimension at position 0
print(f"x shape: {x.shape}")
print(f"x[None] shape: {x[None].shape}")

# Now (x[None] - X) works via broadcasting:
# (1, 2) - (1500, 2) = (1500, 2)
# The (1, 2) tensor is "broadcast" (repeated) to match (1500, 2)
diff_method1 = (x[None] - X)[:8]  # First 8 results
print(f"\nFirst 8 differences using x[None]:")
print(diff_method1)

In [ ]:
# ============================================================================
# SIMPLER BROADCASTING
# ============================================================================

# Actually, (x - X) works directly because PyTorch is smart!
# (2,) - (1500, 2) = (1500, 2)
# The (2,) is automatically broadcast to match
diff_method2 = (x - X)[:8]
print(f"First 8 differences using (x - X) directly:")
print(diff_method2)

print("\nBoth methods give the same result!")
print("Each row shows [x_diff, y_diff] from x to that point")
print("Row 0 is [0, 0] because that's x minus itself")

In [ ]:
# ============================================================================
# COMPUTING EUCLIDEAN DISTANCE
# ============================================================================

# Distance formula: sqrt((x1-x2)^2 + (y1-y2)^2)
# In code:
# 1. (x - X): Differences in each dimension
# 2. **2: Square the differences
# 3. .sum(1): Sum across dimension 1 (the x,y dimension)
# 4. .sqrt(): Take square root

dist = ((x - X)**2).sum(1).sqrt()

print(f"Distances from first point to first 8 points:")
print(dist[:8])
print()
print("Notice: Distance to itself (index 0) is 0")
print("Other distances are typically 3-7 units (depends on cluster spread)")

**What does the code above do?**

Computes Euclidean distance from `x` to every point in `X` in a single line, using exactly the broadcasting pattern we deep-dove on in `01_matmul_explained`:

1. `x - X`: shape `(2,)` minus shape `(1500, 2)`. Broadcasting aligns from the right, treats `x` as shape `(1, 2)`, and produces a `(1500, 2)` tensor of difference vectors — one per data point.
2. `**2`: elementwise square. Still shape `(1500, 2)`.
3. `.sum(1)`: sum along axis 1 (the coordinate axis), reducing to shape `(1500,)` — one squared distance per point.
4. `.sqrt()`: elementwise square root, giving Euclidean distance.

Compare to the explicit loop version that would do the same job:

```python
dist = torch.zeros(1500)
for j in range(1500):
    dist[j] = ((x[0] - X[j, 0])**2 + (x[1] - X[j, 1])**2).sqrt()
```

The vectorized version computes the same 1500 distances ~1000× faster — same idea as the matmul evolution in notebook 01.

In [ ]:
# ============================================================================
# ALTERNATIVE: EUCLIDEAN DISTANCE USING EINSTEIN SUMMATION
# ============================================================================
#
# The original code:
#     dist = ((x - X)**2).sum(1).sqrt()
#
# Can be rewritten using Einstein summation (einsum) for the sum of squares part.
# Let's break this down and show the einsum equivalent.

# First, compute the differences (same as before)
diff = x - X  # Shape: (1500, 2) - difference in x and y for each point

print(f"diff shape: {diff.shape}")
print(f"First few differences: {diff[:5]}")

# ============================================================================
# METHOD 1: Original approach
# ============================================================================
# ((x - X)**2).sum(1).sqrt()
#   - **2: Square each element -> (1500, 2)
#   - .sum(1): Sum along dimension 1 (the x,y coordinates) -> (1500,)
#   - .sqrt(): Square root -> (1500,)

dist_original = ((x - X)**2).sum(1).sqrt()
print(f"Original method: {dist_original[:5]}")

# ============================================================================
# METHOD 2: Einstein summation for the squared sum
# ============================================================================
# 
# einsum('ij,ij->i', diff, diff) means:
#   - 'ij': diff has 2 dimensions (i=points, j=coordinates)
#   - 'ij,ij': Multiply diff with itself element-wise (squaring)
#   - '->i': Sum over j (coordinates), keep i (points)
#
# This computes: sum over j of (diff[i,j] * diff[i,j]) for each i
# Which is exactly: diff[i,0]^2 + diff[i,1]^2 (sum of squared differences)

squared_sum_einsum = torch.einsum('ij,ij->i', diff, diff)
dist_einsum = squared_sum_einsum.sqrt()

print(f"Einsum method:   {dist_einsum[:5]}")

# Verify they're the same
print(f"Distance difference: {torch.max(torch.abs(dist_original - dist_einsum))}")

# ============================================================================
# UNDERSTANDING THE EINSUM NOTATION: 'ij,ij->i'
# ============================================================================
#
# Let's trace through for one point (i=0):
#
#   diff[0] = [dx, dy]  (difference in x and y from point x to X[0])
#
#   einsum('ij,ij->i', diff, diff) at i=0:
#     = diff[0,0]*diff[0,0] + diff[0,1]*diff[0,1]
#     = dx*dx + dy*dy
#     = dx^2 + dy^2
#
#   Then .sqrt() gives: sqrt(dx^2 + dy^2) = Euclidean distance!
#
# The 'j' appears twice in 'ij,ij' but NOT in the output '->i'
# This means: multiply along j, then sum over j (implicit summation)
#
# ============================================================================
# COMPARISON OF APPROACHES
# ============================================================================
#
# Original:  ((x - X)**2).sum(1).sqrt()
#            - More intuitive for beginners
#            - Explicitly shows squaring and summing
#
# Einsum:    torch.einsum('ij,ij->i', diff, diff).sqrt()
#            - More compact
#            - Directly expresses "element-wise multiply and sum"
#            - Useful pattern for dot products and norms
#
# Both produce identical results!

**What does the code above do?**

Recomputes the same distance vector two different ways and verifies they match:

1. **Original method:** `((x - X)**2).sum(1).sqrt()` — broadcasting + reduction.
2. **`einsum` method:** `torch.einsum('ij,ij->i', diff, diff).sqrt()` — both inputs are the same `diff` tensor.

Decoding `'ij,ij->i'` using the rules from the einsum deep dive in notebook 01:

- `i` (axis 0) appears in both inputs and the output → kept as a free axis (one element per data point)
- `j` (axis 1) appears in both inputs but NOT in the output → summed over

So this einsum says: "multiply diff[i, j] by itself, then sum over j" — exactly $\sum_j d_{ij}^2$, the squared distance. After `.sqrt()`, it's the Euclidean distance.

The takeaway: any operation of the form "elementwise multiply two tensors then sum some axes" can be written as a single einsum call. For inner products and norms specifically, einsum is often the most readable option.

In [ ]:
# ============================================================================
# APPLYING THE GAUSSIAN KERNEL TO DISTANCES
# ============================================================================

# Convert distances to weights using Gaussian kernel with bandwidth=2.5
weight = gaussian(dist, 2.5)

print(f"Shape of weights: {weight.shape}")
print(f"First few weights: {weight[:8]}")
print()
print("Notice:")
print("- Weight for distance 0 is ~0.16 (maximum)")
print("- Weights decrease for larger distances")
print("- Very distant points have weight ~0 (shown as 0.000)")

# ============================================================================
# SINGLE POINT UPDATE FUNCTION (SLOW VERSION)
# ============================================================================
# This is the "simple but slow" implementation. It loops through each point
# one at a time. While easy to understand, Python loops are slow!

def one_update(X):
    """
    Perform one Mean Shift iteration on all points.

    Args:
        X: Tensor of shape (n, 2) - all data points
           This tensor is modified IN PLACE (the original changes)

    How the algorithm works for each point:
    =========================================
    
    For each point x in our data:
    
        x ───► Calculate distances to all other points
              │
              ▼
        [d1, d2, d3, ...] ───► Apply kernel to get weights
              │
              ▼
        [w1, w2, w3, ...] ───► Compute weighted average
              │
              ▼
        Move x to the weighted average position
    
    After processing all points, each point has moved toward the local 
    "center of mass" (weighted by the kernel).

    Why is this slow?
    =================
    - Python 'for' loops are slow (interpreted, not compiled)
    - We can't use GPU parallelization effectively
    - Better approach: Process many points at once (batching)
    """
    # Loop through each point (this is the slow part!)
    for i, x in enumerate(X):
        # ------------------------------------------------------------------
        # STEP 1: Compute distances from x to ALL points in X
        # ------------------------------------------------------------------
        # (x - X): Difference vector from each point to x
        # **2: Square each component
        # .sum(1): Sum across x,y coordinates (axis 1)
        # .sqrt(): Take square root to get Euclidean distance
        dist = torch.sqrt(((x - X)**2).sum(1))
        
        # At this point, dist is a 1D tensor of length n (number of points)
        # dist[j] = distance from point x to point X[j]

        # ------------------------------------------------------------------
        # STEP 2: Convert distances to weights using the kernel
        # ------------------------------------------------------------------
        # Nearby points (small distance) get HIGH weights
        # Far points (large distance) get LOW weights (approaching 0)
        # The kernel function controls how quickly weights decrease
        #
        # Using triangular kernel with reach=8
        # (You could also use: weight = gaussian(dist, 2.5))
        weight = tri(dist, 8)
        
        # weight is also a 1D tensor of length n
        # weight[j] = how much point X[j] influences where x should move

        # ------------------------------------------------------------------
        # STEP 3: Compute weighted average of all points
        # ------------------------------------------------------------------
        # Numerator: Sum of (weight × point) for each point
        #   weight[:, None] reshapes (n,) to (n, 1) for broadcasting
        #   (n, 1) * (n, 2) = (n, 2) - each point multiplied by its weight
        #   .sum(0) sums across all points, giving (2,) - the weighted sum
        numerator = (weight[:, None] * X).sum(0)
        
        # Denominator: Sum of all weights (for normalization)
        denominator = weight.sum()

        # ------------------------------------------------------------------
        # STEP 4: Update the point to the weighted average position
        # ------------------------------------------------------------------
        # This moves x toward the local "center of gravity"
        # Points in dense regions move toward cluster centers
        # Points on the edge move inward
        X[i] = numerator / denominator

In [ ]:
# ============================================================================
# COMPUTING WEIGHTED POINTS
# ============================================================================

# Multiply each point by its weight
weighted_X = weight[:, None] * X

print("First few weighted points:")
print(weighted_X[:5])
print()
print("Notice how points with higher weights keep their values,")
print("while points with lower weights become nearly zero.")

**What does the code above do?**

`weight[:, None] * X` multiplies each point in `X` by its corresponding weight using broadcasting:

- `weight` has shape `(1500,)`
- `weight[:, None]` has shape `(1500, 1)`
- `X` has shape `(1500, 2)`
- `(1500, 1) * (1500, 2)` → broadcasting stretches the `1` to `2` → shape `(1500, 2)`

After this multiplication, points with high weight contribute their full coordinates; points with weight ≈ 0 contribute ≈ 0. Summing this along the point axis would give the *unnormalized* weighted center of mass — exactly the numerator of the mean-shift update.

### Putting It All Together: The Update Step

Now we can write the complete update function that moves all points one step toward their local density centers.

In [ ]:
# ============================================================================
# SINGLE POINT UPDATE FUNCTION (SLOW VERSION)
# ============================================================================
# This is the "simple but slow" implementation - perfect for learning!
# It processes one point at a time using Python loops.

def one_update(X):
    """
    Perform one Mean Shift iteration on all points.
    
    What this function does:
    ========================
    For EACH point in our dataset:
    1. Calculate how far it is from every other point
    2. Convert those distances into "influence weights" (closer = more influence)
    3. Compute a weighted average position (where should this point move?)
    4. Move the point to that new position
    
    After calling this function once, all points will have moved slightly
    toward their local "centers of mass". Call it multiple times and
    points will converge to cluster centers!

    Args:
        X: Tensor of shape (n, 2) - all data points
           IMPORTANT: This tensor is modified IN PLACE (the original changes!)
    
    Why is this version slow?
    =========================
    - Python 'for' loops are interpreted (not compiled) - slow!
    - We process points one at a time instead of in parallel
    - We can't effectively use GPU acceleration
    - For 1500 points: 1500 loop iterations × complex calculations = slow
    """
    # ══════════════════════════════════════════════════════════════════════
    # MAIN LOOP: Process each point one at a time
    # ══════════════════════════════════════════════════════════════════════
    # enumerate gives us both the index (i) and the point (x)
    # i = 0, 1, 2, ... (which point we're updating)
    # x = the actual [x, y] coordinates of that point
    
    for i, x in enumerate(X):
        # ══════════════════════════════════════════════════════════════════
        # STEP 1: Compute distances from point x to ALL points in X
        # ══════════════════════════════════════════════════════════════════
        # 
        # Mathematical formula: distance = sqrt((x1-x2)² + (y1-y2)²)
        #
        # In code:
        #   (x - X)     → Difference vectors: [[x-X0_x, y-X0_y], [x-X1_x, y-X1_y], ...]
        #                 Shape: (n, 2) where n = number of points
        #   **2         → Square each element: [[diff_x², diff_y²], ...]
        #   .sum(1)     → Sum across axis 1 (the x,y dimension): [diff_x² + diff_y², ...]
        #                 Shape: (n,) - one squared distance per point
        #   .sqrt()     → Square root to get actual distances
        #                 Shape: (n,) - one distance per point
        #
        dist = torch.sqrt(((x - X)**2).sum(1))
        
        # After this line:
        # dist[0] = distance from x to X[0]
        # dist[1] = distance from x to X[1]
        # ... and so on for all n points
        # Note: dist[i] will be 0 (distance from x to itself)

        # ══════════════════════════════════════════════════════════════════
        # STEP 2: Convert distances to weights using the kernel function
        # ══════════════════════════════════════════════════════════════════
        #
        # The kernel function answers: "How much should each point influence
        # where x moves to?"
        #
        # - Nearby points (small distance) → HIGH weight → strong pull
        # - Far points (large distance) → LOW weight → weak pull
        # - Very far points → weight ≈ 0 → essentially no influence
        #
        # Using triangular kernel with reach=8:
        # - At distance 0: weight = 1.0 (maximum influence)
        # - At distance 4: weight = 0.5 (half influence)
        # - At distance 8: weight = 0.0 (no influence)
        # - Beyond 8: weight = 0.0 (completely ignored)
        #
        weight = tri(dist, 8)
        
        # Alternative: Use Gaussian kernel for smoother results
        # weight = gaussian(dist, 2.5)

        # ══════════════════════════════════════════════════════════════════
        # STEP 3: Compute weighted average of all points
        # ══════════════════════════════════════════════════════════════════
        #
        # Formula: new_position = Σ(weight_i × point_i) / Σ(weight_i)
        #
        # This is like asking: "If each point pulls on x with strength
        # proportional to its weight, where does x end up?"
        #
        # Breaking down the numerator calculation:
        #   weight          → Shape: (n,) - one weight per point
        #   weight[:, None] → Shape: (n, 1) - add dimension for broadcasting
        #   X               → Shape: (n, 2) - all points
        #   weight[:, None] * X → Shape: (n, 2) - each point scaled by its weight
        #   .sum(0)         → Shape: (2,) - sum all weighted points
        #
        numerator = (weight[:, None] * X).sum(0)
        
        # Denominator is just the sum of all weights (for normalization)
        denominator = weight.sum()

        # ══════════════════════════════════════════════════════════════════
        # STEP 4: Update the point's position
        # ══════════════════════════════════════════════════════════════════
        #
        # Move point i to its new position (the weighted average)
        # This shifts x toward where the density of nearby points is highest
        #
        X[i] = numerator / denominator
        
    # After this function completes:
    # - Every point has moved toward its local "center of mass"
    # - Points in dense regions barely move (they're already near center)
    # - Points on edges move toward the dense center
    # - After multiple iterations, points converge to cluster centers

**What does the code above do?**

Defines `one_update` — the function that performs one iteration of mean shift on *every* point in `X`. For each point, it:

1. Computes distances to all 1500 points.
2. Converts distances to triangular-kernel weights (reach 8).
3. Computes the weighted average position.
4. Updates the point in place.

**This is the slow version on purpose.** It loops in Python over every point, which is exactly what we'll vectorize away in §7. Understanding the slow version first is essential because it makes the speedup story concrete — when we replace the Python loop with batched tensor operations, you'll know exactly what the loop was doing.

**Subtle gotcha — in-place vs out-of-place updates.** The line `X[i] = numerator / denominator` modifies `X` *while still iterating over it*. This means when point `i+1` computes its distances, it uses the *updated* position of point `i`, not the original. This is a deliberate choice — it makes the algorithm converge slightly faster than the "simultaneous update" version where you'd compute all new positions before assigning. The difference is small in practice and either version converges to roughly the same clusters.

In [ ]:
# ============================================================================
# COMPLETE MEAN SHIFT ALGORITHM (SLOW VERSION)
# ============================================================================

def meanshift(data):
    """
    Run Mean Shift clustering algorithm.

    Args:
        data: Tensor of shape (n, 2) - all data points

    Returns:
        X: Tensor of shape (n, 2) - points after clustering
            (nearby points will have converged to similar locations)
    """
    # Make a copy so we don't modify the original data
    X = data.clone()

    # Run 5 iterations of the algorithm
    # (Usually 5-10 iterations is enough for convergence)
    for it in range(5):
        one_update(X)
        print(f"Completed iteration {it + 1}")

    return X

**What does the code above do?**

Wraps `one_update` in a loop that runs five iterations. Five is usually enough to see clear cluster convergence on synthetic data like ours — points in dense regions barely move after iteration 2 or 3, but the boundary points need a few more passes to converge.

`.clone()` at the top is important: without it, we'd mutate the original `data` tensor. With it, the function is pure from the caller's perspective.

There's no convergence check — we just hardcode 5 iterations. In production you'd typically iterate until point movements drop below a tolerance, but for teaching purposes the fixed loop is clearer.

In [ ]:
# ============================================================================
# RUNNING MEAN SHIFT (SLOW VERSION)
# ============================================================================

# Time how long it takes
%time X = meanshift(data)

print()
print("Mean Shift complete!")
print("Points within each cluster should now be very close together.")

**What does the code above do?**

Runs the sequential mean shift on all 1500 points and times it.

Expect this to be slow — somewhere between 500 ms and several seconds depending on your CPU. That's 1500 points × 5 iterations × (one Python loop iteration per point) = 7,500 Python-loop iterations, each doing some tensor work. The tensor work itself is fast, but the Python loop overhead dominates.

This timing is the baseline that §7's batched version will improve by ~100–200× on a GPU.

In [ ]:
# ============================================================================
# VISUALIZING THE RESULTS
# ============================================================================

# Plot the result
# centroids+2 shifts the X markers slightly so we can see both
# original centroids and where points converged
plot_data(centroids+2, X, n_samples)
plt.title("After Mean Shift Clustering\n(Points converged toward cluster centers)")
plt.show()

print("Success! Points from each cluster have converged to nearly the same location.")
print("The X markers show the original centroids (shifted for visibility).")

---

# Deep Dive: Why Mean Shift Converges

We've watched mean shift converge in the plot. But *why* does it converge? What's the math that guarantees points move toward density peaks rather than drifting randomly or oscillating?

The answer is one of the most elegant results in unsupervised learning: **mean shift is gradient ascent on a kernel density estimate.** Once you see this, the entire algorithm — including the choice of Gaussian kernel, the weighted-average update, and the fact that it converges to local maxima — falls out of one short derivation.

---

## The Setup

We have $n$ data points $x_1, \ldots, x_n$ in $\mathbb{R}^d$. The kernel density estimate of the underlying distribution, with Gaussian kernel and bandwidth $\sigma$, is:

$$\hat{p}(y) = \frac{1}{n (2\pi \sigma^2)^{d/2}} \sum_{i=1}^{n} \exp\!\left(-\frac{\|y - x_i\|^2}{2\sigma^2}\right).$$

This $\hat{p}$ is a smooth function on $\mathbb{R}^d$ — peaked wherever data is dense, low wherever data is sparse. We want to find its *local maxima* (the "modes"), because those are the natural cluster centers.

The natural way to climb a smooth function is **gradient ascent**: from the current position $y$, take a small step in the direction $\nabla \hat{p}(y)$.

---

## Computing the Gradient

Take the gradient of $\hat{p}$ with respect to $y$. The leading constant comes out front; the chain rule on each exponential gives:

$$\nabla \hat{p}(y) = \frac{1}{n (2\pi\sigma^2)^{d/2}} \sum_{i=1}^{n} \exp\!\left(-\frac{\|y - x_i\|^2}{2\sigma^2}\right) \cdot \nabla\!\left(-\frac{\|y - x_i\|^2}{2\sigma^2}\right).$$

The gradient of $-\|y - x_i\|^2 / (2\sigma^2)$ with respect to $y$ is $-(y - x_i)/\sigma^2 = (x_i - y)/\sigma^2$. Pulling out the $1/\sigma^2$:

$$\nabla \hat{p}(y) = \frac{1}{n \sigma^2 (2\pi\sigma^2)^{d/2}} \sum_{i=1}^{n} \exp\!\left(-\frac{\|y - x_i\|^2}{2\sigma^2}\right) \,(x_i - y).$$

Let's introduce a shorthand for the Gaussian kernel weight at point $x_i$ given current position $y$:

$$w_i(y) \;=\; \exp\!\left(-\frac{\|y - x_i\|^2}{2\sigma^2}\right).$$

Then:

$$\nabla \hat{p}(y) \;=\; \frac{1}{n \sigma^2 (2\pi\sigma^2)^{d/2}} \sum_{i=1}^{n} w_i(y) \,(x_i - y).$$

---

## The Mean-Shift Vector

Split the sum: $\sum_i w_i (x_i - y) = \sum_i w_i x_i - y \sum_i w_i$. So:

$$\sum_i w_i (x_i - y) \;=\; \left(\sum_i w_i\right)\!\left[\frac{\sum_i w_i x_i}{\sum_i w_i} - y\right].$$

Look at the bracket. The first term is the **weighted mean** of the data points with kernel weights $w_i$ — exactly what one iteration of mean shift computes! Call it $m(y)$:

$$m(y) \;=\; \frac{\sum_i w_i(y)\, x_i}{\sum_i w_i(y)}.$$

Substituting back:

$$\nabla \hat{p}(y) \;=\; \frac{\sum_i w_i(y)}{n \sigma^2 (2\pi\sigma^2)^{d/2}} \cdot \left[\,m(y) - y\,\right].$$

The gradient of $\hat{p}$ at $y$ is **proportional to the vector $m(y) - y$** — the direction from the current point to the weighted mean of nearby points.

The leading coefficient $\frac{\sum_i w_i(y)}{n \sigma^2 (2\pi\sigma^2)^{d/2}}$ is always positive (it's a sum of positive Gaussian weights times positive constants), so:

$$\text{direction of }\nabla \hat{p}(y) \;=\; \text{direction of }[m(y) - y].$$

---

## What Mean Shift Actually Does

The mean-shift update is:

$$y_{\text{new}} \;=\; m(y) \;=\; y + [m(y) - y].$$

That is: **step from $y$ in the direction $m(y) - y$ by exactly that vector's length.** We just showed $m(y) - y$ is in the direction of $\nabla \hat{p}(y)$. So mean shift is taking a step in the direction of the gradient of the kernel density estimate.

This is gradient ascent with a particular, automatically-chosen step size — proportional to $1 / \sum_i w_i$. The step size is *adaptive*: in dense regions (lots of nearby points → large $\sum_i w_i$), the step is *small*; in sparse regions, the step is large. This is desirable — fine-grained updates near peaks, big jumps when far from any structure.

---

## Why It Converges

Two facts together give convergence:

1. **The step direction is always uphill.** Because $m(y) - y$ is parallel to $\nabla \hat{p}(y)$, taking the step strictly increases $\hat{p}$ (for non-stationary points). Each iteration moves us toward higher density.

2. **The density estimate $\hat{p}$ is bounded above.** A finite sum of bounded Gaussians can't exceed some finite ceiling. A monotone sequence (the values $\hat{p}(y_t)$) bounded above must converge.

So the value $\hat{p}(y_t)$ converges. With some standard arguments about smooth functions, the position $y_t$ also converges, and it converges to a **stationary point of $\hat{p}$** — that is, a point where $\nabla \hat{p} = 0$. For density estimates these are local maxima, saddle points, or local minima; in practice, the only stable attractors are local maxima. So **mean shift converges to a local maximum of the kernel density estimate.**

That's the entire theory. The algorithm finds modes of the density, and the modes are the cluster centers.

---

## The Big Picture

What looked like a heuristic — "compute the weighted mean of nearby points and move there" — is actually a principled optimization algorithm:

| Heuristic view | Mathematical view |
|---|---|
| Move each point toward the weighted mean of its neighbors. | Gradient ascent on $\hat{p}$, the kernel density estimate. |
| Bandwidth controls neighborhood size. | Bandwidth controls smoothness of $\hat{p}$. |
| Points converge to "dense regions." | Points converge to local maxima of $\hat{p}$. |
| The Gaussian kernel is the obvious choice. | The Gaussian kernel makes the gradient calculation clean and isotropic. |

And the algorithm needs *no* hyperparameter for the number of clusters because the modes of $\hat{p}$ are intrinsic features of the data and the bandwidth choice — once those are fixed, the modes are determined.

The deep dive on attention in notebook 27 will reveal one more layer: the weighted-average operation `(weights @ X) / weights.sum(1, keepdim=True)` from the batched mean shift in §7 is *structurally identical* to the value-aggregation step of softmax attention. Same shapes, same operations, different interpretation. The next deep dive walks through that connection.

---
## Part 4: GPU-Accelerated Batched Algorithm

### Why is the simple version slow?

Our simple implementation has two problems:
1. **Python loops are slow**: We loop through each point one at a time
2. **No GPU acceleration**: All computation happens on CPU

### The solution: Batching

Instead of updating one point at a time, we update **batches of points simultaneously**. This is perfect for GPUs, which excel at parallel computation.

### Visual Explanation: Sequential vs Batched Processing

```
SEQUENTIAL (Slow - what we did before):
┌──────────────────────────────────────────────────┐
│ Time →                                           │
│                                                  │
│ Point 1: [compute]                               │
│ Point 2:          [compute]                      │
│ Point 3:                   [compute]             │
│ Point 4:                            [compute]    │
│                                                  │
│ Total time = 4 × (time for one point)           │
└──────────────────────────────────────────────────┘

BATCHED (Fast - GPU can do all at once):
┌──────────────────────────────────────────────────┐
│ Time →                                           │
│                                                  │
│ Point 1: ┐                                       │
│ Point 2: ├─[compute all together]                │
│ Point 3: │                                       │
│ Point 4: ┘                                       │
│                                                  │
│ Total time ≈ (time for one point) ← HUGE WIN!   │
└──────────────────────────────────────────────────┘
```

### Connection to Deep Learning

**Batching** is fundamental to deep learning:
- Training always uses **mini-batches** of data
- Stable Diffusion processes batches of latent vectors
- GPUs are optimized for parallel batch operations

**Why GPUs love batching:**
- GPUs have thousands of cores (vs ~8 for CPUs)
- Each core can handle one calculation
- Batching lets us use ALL cores simultaneously

Let's implement a batched version!

In [ ]:
# ============================================================================
# IMPORTING ANIMATION TOOLS
# ============================================================================

from matplotlib.animation import FuncAnimation  # For creating animations
from IPython.display import HTML                 # For displaying in Jupyter

# FuncAnimation creates animations by repeatedly calling a function
# HTML allows us to display the animation in the notebook

In [ ]:
### Batched Distance Computation

Computing distances for a batch is trickier. We need distances from each batch point to all data points.

**Shape goal**: (batch_size, n_points) = (5, 1500)

This requires careful use of broadcasting with extra dimensions.

### 3D Broadcasting Visualization

When computing batched distances, we use 3D tensors. Here's how it works:

```
We have:
  X = all points, shape (1500, 2)      ← 1500 points, each with x,y
  x = batch of query points, shape (5, 2)  ← 5 points we're computing for

We want:
  distances of shape (5, 1500)         ← distance from each of 5 to each of 1500

The trick: Add dimensions to create a 3D "grid" of differences!

Step 1: Reshape for broadcasting
  X[None]    → shape (1, 1500, 2)      ← Add batch dimension
  x[:, None] → shape (5, 1, 2)         ← Add points dimension

Step 2: Broadcasting creates all pairwise differences
  (1, 1500, 2) - (5, 1, 2) → (5, 1500, 2)
  
  This 3D tensor contains:
  ┌─────────────────────────────────────┐
  │ Layer 0: Differences for query 0    │
  │   Point 0: [x0-X0_x, y0-X0_y]       │
  │   Point 1: [x0-X1_x, y0-X1_y]       │
  │   ...      (1500 such pairs)        │
  ├─────────────────────────────────────┤
  │ Layer 1: Differences for query 1    │
  │   ... (same structure)              │
  ├─────────────────────────────────────┤
  │ ... (5 layers total, one per query) │
  └─────────────────────────────────────┘

Step 3: Square, sum over coordinates, sqrt
  (5, 1500, 2) → square → (5, 1500, 2)
              → sum(axis=2) → (5, 1500)
              → sqrt → (5, 1500) ✓
```

In [ ]:
# ============================================================================
# CREATE AND DISPLAY THE ANIMATION
# ============================================================================

# Reset X to original data
X = data.clone()

# Create figure and axis for animation
fig, ax = plt.subplots()

# Create animation
# - fig: The figure to animate
# - do_one: Function to call for each frame
# - frames=5: Number of frames (iterations)
# - interval=500: Milliseconds between frames
# - repeat=False: Don't loop the animation
ani = FuncAnimation(fig, do_one, frames=5, interval=500, repeat=False)

# Close the static figure (we'll display the animation instead)
plt.close()

# Display as HTML video
HTML(ani.to_jshtml())

In [ ]:
# ============================================================================
# SETTING UP FOR BATCHED COMPUTATION
# ============================================================================

bs = 5  # Batch size - process 5 points at a time
X = data.clone()

# Take first batch of 5 points
x = X[:bs]

print(f"Batch shape: {x.shape}")      # (5, 2) - 5 points, each with x,y
print(f"Full data shape: {X.shape}")   # (1500, 2) - all points

print()
print("Instead of processing 1 point at a time,")
print("we'll process 5 (or more) points simultaneously!")

In [ ]:
# ============================================================================
# BATCHED DISTANCE FUNCTION - THE KEY TO GPU SPEEDUP!
# ============================================================================
# This function computes distances from MANY query points to ALL data points
# at once, using clever broadcasting tricks.

def dist_b(a, b):
    """
    Compute distances from each point in b to all points in a.
    
    This is the BATCHED version - it handles multiple query points at once!

    Args:
        a: All data points, shape (n, 2)
           Example: 1500 points, each with [x, y] coordinates
        b: Batch of query points, shape (bs, 2)
           Example: 5 points we want to compute distances for

    Returns:
        Distances, shape (bs, n)
        Example: (5, 1500) = distance from each of 5 queries to each of 1500 points
        
    Visual representation of the output:
    ====================================
                        All 1500 data points
                    Point0  Point1  Point2  ...  Point1499
    Query0      [   d0,0    d0,1    d0,2   ...   d0,1499  ]
    Query1      [   d1,0    d1,1    d1,2   ...   d1,1499  ]
    Query2      [   d2,0    d2,1    d2,2   ...   d2,1499  ]
    Query3      [   d3,0    d3,1    d3,2   ...   d3,1499  ]
    Query4      [   d4,0    d4,1    d4,2   ...   d4,1499  ]
    
    Where d[i,j] = distance from query point i to data point j
    """
    # ══════════════════════════════════════════════════════════════════════
    # THE BROADCASTING TRICK - Computing all pairwise differences at once!
    # ══════════════════════════════════════════════════════════════════════
    #
    # We want to compute (b[i] - a[j]) for ALL combinations of i and j
    # That's bs × n = 5 × 1500 = 7,500 difference vectors!
    #
    # Step 1: Reshape tensors for broadcasting
    #   a has shape (n, 2) = (1500, 2)
    #   a[None] adds a dimension at the start: (1, n, 2) = (1, 1500, 2)
    #   
    #   b has shape (bs, 2) = (5, 2)
    #   b[:, None] adds a dimension in the middle: (bs, 1, 2) = (5, 1, 2)
    #
    # Step 2: Broadcasting magic!
    #   (1, 1500, 2) - (5, 1, 2) = ???
    #   
    #   PyTorch aligns dimensions from the right:
    #   - Dimension 2: Both are 2 ✓ (coordinates)
    #   - Dimension 1: 1500 vs 1 → broadcast to 1500
    #   - Dimension 0: 1 vs 5 → broadcast to 5
    #   
    #   Result shape: (5, 1500, 2) - ALL pairwise differences!
    #
    # Visualization of what happens:
    #   a[None] "copies" a for each query point (virtually, not in memory)
    #   b[:, None] "copies" each query for all data points (virtually)
    #   The subtraction happens element-wise on this "virtual" 3D grid
    
    diff = a[None] - b[:, None]  # Shape: (bs, n, 2) = (5, 1500, 2)
    
    # diff[i, j, :] = difference vector from query i to data point j
    # diff[i, j, 0] = x-coordinate difference
    # diff[i, j, 1] = y-coordinate difference

    # ══════════════════════════════════════════════════════════════════════
    # COMPUTE EUCLIDEAN DISTANCE: sqrt(dx² + dy²)
    # ══════════════════════════════════════════════════════════════════════
    
    squared = diff ** 2          # Square each component
                                  # Shape: (5, 1500, 2) - squared differences
                                  
    summed = squared.sum(2)      # Sum across last dimension (coordinates)
                                  # Shape: (5, 1500) - squared distances
                                  # sum(2) means "sum along dimension 2"
                                  
    distances = summed.sqrt()    # Take square root
                                  # Shape: (5, 1500) - actual distances

    return distances

# ════════════════════════════════════════════════════════════════════════════
# WHY IS THIS FASTER?
# ════════════════════════════════════════════════════════════════════════════
# 
# Python loop version (slow):
#   for i in range(5):
#       for j in range(1500):
#           distances[i,j] = compute_distance(b[i], a[j])
#   → 7,500 Python loop iterations!
#
# Batched version (fast):
#   distances = dist_b(a, b)
#   → ONE operation that PyTorch optimizes and can run on GPU!
#
# On GPU, all 7,500 distance calculations happen IN PARALLEL!

**What does the code above do?**

Defines `dist_b(a, b)` — the **batched pairwise distance function** that lets us update many points at once. This is the structural heart of the GPU-accelerated mean shift, so let me walk through the broadcasting carefully:

- `a` has shape `(n, 2)` — all data points (n = 1500 in our case).
- `b` has shape `(bs, 2)` — the batch of query points we want distances for (bs = e.g. 500).

We want a tensor of shape `(bs, n)` where entry `[i, j]` is the distance from `b[i]` to `a[j]`. The broadcasting trick:

| Tensor | Shape after reshape | What each axis means |
|---|---|---|
| `a[None]` | `(1, n, 2)` | (batch, point, coord). The `1` lets it broadcast over batches. |
| `b[:, None]` | `(bs, 1, 2)` | (batch, point, coord). The `1` lets it broadcast over points. |
| `a[None] - b[:, None]` | `(bs, n, 2)` | All `bs × n` pairwise difference vectors. |
| `.sum(2)` after `**2` | `(bs, n)` | Sum over the coordinate axis → squared distances. |
| `.sqrt()` | `(bs, n)` | Euclidean distances. |

This is **exactly** the pattern that scaled dot-product attention uses to compute query–key similarities (transformers compute `Q @ K.T` to get a `(batch, q_len, k_len)` similarity tensor). Mean shift and attention are doing the same shape gymnastics, with different intent.

In [ ]:
# ============================================================================
# TESTING BATCHED DISTANCE
# ============================================================================

# Compute distances from first 5 points to all 1500 points
distances = dist_b(X, x)

print(f"Batched distances shape: {distances.shape}")
print()
print("First 5 columns (distances to first 5 points in X):")
print(distances[:, :5])
print()
print("Notice the diagonal is 0 (distance to self)")
print("This computed 5 × 1500 = 7,500 distances in one operation!")

In [ ]:
# ============================================================================
# METHOD 2: EINSTEIN SUMMATION (EINSUM)
# ============================================================================
# einsum is a VERY powerful tool for tensor operations. Once you learn it,
# you can express complex operations in a single line!
#
# The notation 'ij,jk->ik' tells einsum:
#   - The inputs have indices 'ij' (weight) and 'jk' (X)
#   - The output has indices 'ik'
#   - Any index NOT in the output is summed over (here, 'j')
#
# What each letter represents:
#   i = batch dimension (5 query points)
#   j = points dimension (1500 data points)
#   k = coordinate dimension (2: x and y)
#
# In plain English: "For each query point i and coordinate k,
#                   sum over all data points j:  weight[i,j] * X[j,k]"

# Breaking down the operation:
#   weight[i,j] is a scalar: the weight for query i and point j
#   X[j,k] is the k-th coordinate of point j
#   We multiply and sum over j, getting result for (i, k)

num_einsum = torch.einsum('ij,jk->ik', weight, X)

print("Using einsum notation: 'ij,jk->ik'")
print("  - i loops over batch (5)")
print("  - j loops over points (1500) - summed over")
print("  - k loops over coordinates (2)")
print()
print(f"Result shape: {num_einsum.shape}")
print(f"Result values:\n{num_einsum}")
print()
print("Same as Method 1! einsum is compact but takes practice to read.")

**What does the code above do?**

Computes the batched weighted average using `einsum`, as an alternative formulation of the matmul we'll see two cells later. The einsum string `'ij,jk->ik'`:

- `i` = batch index (5 query points)
- `j` = data point index (1500 — summed over)
- `k` = coordinate index (2 — preserved)

`weight[i, j] * X[j, k]` is multiplied for every triple, then summed over `j`. Result: `(5, 2)` — one weighted-average position per query.

This is identical to matrix multiplication `weight @ X`. The next cell does it that way. Both produce the same numbers; einsum's advantage is when you want to express more elaborate contractions (batched attention, bilinear forms, etc.).

In [ ]:
# ============================================================================
# BATCHED WEIGHTS
# ============================================================================

# Apply Gaussian kernel to all distances at once
weight = gaussian(dist_b(X, x), 2)

print(f"Weight shape: {weight.shape}")  # (5, 1500)
print()
print("First few weights for each batch point:")
print(weight[:, :5])

### Batched Weighted Average

Now we need to compute the weighted average for each point in the batch. There are multiple ways to do this:

1. **Broadcasting with extra dimensions**
2. **Einstein summation (einsum)**
3. **Matrix multiplication (@)**

Let's explore all three!

In [ ]:
# ============================================================================
# METHOD 1: BROADCASTING WITH EXTRA DIMENSIONS
# ============================================================================

print(f"weight shape: {weight.shape}")      # (5, 1500)
print(f"X shape: {X.shape}")                 # (1500, 2)

# We need to multiply each weight by each point's coordinates
# weight[..., None] adds a dimension: (5, 1500) -> (5, 1500, 1)
# X[None] adds a dimension: (1500, 2) -> (1, 1500, 2)

print(f"weight[..., None] shape: {weight[..., None].shape}")  # (5, 1500, 1)
print(f"X[None] shape: {X[None].shape}")                       # (1, 1500, 2)

# Multiplication broadcasts: (5, 1500, 1) * (1, 1500, 2) = (5, 1500, 2)
# Then sum over the 1500 points (dimension 1)
num = (weight[..., None] * X[None]).sum(1)
print(f"\nNumerator shape: {num.shape}")  # (5, 2)
print(f"Numerator values:\n{num}")

**What does the code above do?**

The "manual broadcasting" version of the same weighted average:

- `weight[..., None]` has shape `(5, 1500, 1)` — adds a trailing dim for the coordinate axis to broadcast over.
- `X[None]` has shape `(1, 1500, 2)` — adds a leading dim for the batch axis to broadcast over.
- Multiplying gives shape `(5, 1500, 2)` — every weighted point.
- `.sum(1)` reduces the 1500-axis, leaving `(5, 2)` — one weighted-sum per batch query.

This is the most explicit form of what's happening. The next cells will show that `einsum('ij,jk->ik', weight, X)` and `weight @ X` produce the exact same result, but this manual broadcasting makes the shape mechanics visible.

In [ ]:
# ============================================================================
# FAST BATCHED MEAN SHIFT IMPLEMENTATION
# ============================================================================
# This is the optimized version that can run on GPU. It's the same algorithm
# as one_update(), but processes many points at once for massive speedup!

def meanshift(data, bs=500):
    """
    GPU-accelerated Mean Shift clustering.

    Args:
        data: Tensor of shape (n, 2) - all data points
              Can be on CPU or GPU (use data.cuda() for GPU)
        bs: Batch size - how many points to update simultaneously
            Larger bs = faster, but uses more memory
            Typical values: 250-1000, depending on your GPU memory

    Returns:
        X: Tensor of shape (n, 2) - clustered points
           Points from the same cluster will converge to similar locations

    The key optimizations:
        1. Process 'bs' points at once instead of 1
        2. Use matrix operations (@) instead of loops  
        3. All operations are GPU-compatible

    Memory consideration:
        The intermediate distance matrix has shape (bs, n)
        For n=1500 and bs=500: 750,000 floats ≈ 3 MB
        For n=100,000 and bs=500: 50,000,000 floats ≈ 200 MB
    """
    n = len(data)
    
    # Make a copy so we don't modify the original data
    # This is important - we update positions iteratively
    X = data.clone()

    # Run 5 iterations (usually enough for convergence)
    # You could add a convergence check instead of fixed iterations
    for it in range(5):
        
        # Process the data in batches
        # range(0, n, bs) gives: 0, bs, 2*bs, 3*bs, ...
        for i in range(0, n, bs):
            
            # Create a slice object for the current batch
            # slice(i, min(i+bs, n)) handles the last batch which might be smaller
            # Example: if n=1500, bs=500:
            #   Batch 0: slice(0, 500)
            #   Batch 1: slice(500, 1000)  
            #   Batch 2: slice(1000, 1500)
            s = slice(i, min(i + bs, n))

            # ══════════════════════════════════════════════════════════
            # STEP 1: Compute distances from batch to all points
            # ══════════════════════════════════════════════════════════
            # X[s] has shape (bs, 2) - the batch of points we're updating
            # X has shape (n, 2) - all points
            # distances has shape (bs, n) - distance from each batch point to all points
            distances = dist_b(X, X[s])

            # ══════════════════════════════════════════════════════════
            # STEP 2: Convert distances to weights
            # ══════════════════════════════════════════════════════════
            # Using Gaussian kernel with bandwidth 2.5
            # weight[i, j] = how much point j influences batch point i
            weight = gaussian(distances, 2.5)

            # ══════════════════════════════════════════════════════════
            # STEP 3: Compute weighted average using matrix multiplication
            # ══════════════════════════════════════════════════════════
            # This is the magic line! Matrix multiplication computes all
            # weighted sums at once:
            #   weight @ X 
            #   (bs, n) @ (n, 2) = (bs, 2)
            # Each row of the result is the weighted sum for one batch point
            numerator = weight @ X
            
            # Sum of weights for each batch point
            # keepdim=True keeps shape (bs, 1) for broadcasting in division
            denominator = weight.sum(1, keepdim=True)

            # ══════════════════════════════════════════════════════════
            # STEP 4: Update batch positions
            # ══════════════════════════════════════════════════════════
            # (bs, 2) / (bs, 1) = (bs, 2) via broadcasting
            X[s] = numerator / denominator

    return X

**What does the code above do?**

The batched mean shift, end to end. For each iteration (5 of them) and each batch (n / bs of them):

1. **Compute distances** — `dist_b(X, X[s])` produces a `(bs, n)` distance matrix.
2. **Apply the kernel** — `gaussian(distances, 2.5)` gives a `(bs, n)` weight matrix.
3. **Compute new positions** — `weight @ X` does the entire weighted-sum-over-points operation as a single matmul. Divide by row sums to normalize. Assign back to `X[s]`.

The key move is replacing the Python loop over points with a Python loop over *batches*. With `bs=500` and `n=1500`, we have 3 batches per iteration × 5 iterations = 15 Python loop iterations — versus the slow version's 1500 × 5 = 7,500. And each batch operation does dramatically more work in vectorized C/CUDA per loop iteration.

In [ ]:
# ============================================================================
# METHOD 3: MATRIX MULTIPLICATION (@)
# ============================================================================

# The @ operator does matrix multiplication
# (5, 1500) @ (1500, 2) = (5, 2)
# This is exactly what we need!

num_matmul = weight @ X

print(f"Result shape: {num_matmul.shape}")
print(f"Result values:\n{num_matmul}")
print()
print("Same result again! Matrix multiplication is the cleanest way.")

**What does the code above do?**

The same weighted average using `@` (the matrix multiplication operator). Shape arithmetic:

- `weight` is `(5, 1500)`.
- `X` is `(1500, 2)`.
- `weight @ X` contracts on the shared 1500 axis, giving `(5, 2)`.

`(weight @ X)[i, k] = sum_j weight[i, j] * X[j, k]` — exactly the weighted sum. This is the most idiomatic PyTorch form and the one to use in production.

In [ ]:
# ============================================================================
# COMPUTING THE DENOMINATOR AND FINAL RESULT
# ============================================================================

# Denominator is sum of weights for each batch point
# keepdim=True keeps the shape compatible for division
div = weight.sum(1, keepdim=True)

print(f"Denominator shape: {div.shape}")  # (5, 1)
print(f"Denominator values:\n{div}")

# Final weighted average
new_positions = num_matmul / div

print(f"\nNew positions shape: {new_positions.shape}")
print(f"New positions:\n{new_positions}")
print()
print("These are where the 5 batch points should move to!")

**What does the code above do?**

Completes the batched mean-shift update by dividing by the sum of weights per query:

- `weight.sum(1, keepdim=True)` has shape `(5, 1)` — the `keepdim` preserves the trailing axis so the division broadcasts cleanly over coordinates.
- `num_matmul / div` is `(5, 2) / (5, 1)` → broadcasts to `(5, 2)` — the new positions.

Five points have just been updated in a single tensor expression. The same expression would update 5,000 or 50,000 points just by changing the slice — no loop changes needed.

### Complete GPU-Accelerated Mean Shift

Now let's put it all together into a fast, batched implementation.

In [ ]:
# ============================================================================
# FAST BATCHED MEAN SHIFT IMPLEMENTATION
# ============================================================================

def meanshift(data, bs=500):
    """
    GPU-accelerated Mean Shift clustering.

    Args:
        data: Tensor of shape (n, 2) - all data points
        bs: Batch size - how many points to update simultaneously

    Returns:
        X: Tensor of shape (n, 2) - clustered points

    The key optimizations:
        1. Process points in batches of size bs
        2. Use matrix operations instead of loops
        3. Everything can run on GPU if data is on GPU
    """
    n = len(data)
    X = data.clone()

    # Run 5 iterations
    for it in range(5):
        # Process in batches
        for i in range(0, n, bs):
            # Create slice for current batch
            s = slice(i, min(i + bs, n))

            # Compute distances from batch to all points
            distances = dist_b(X, X[s])

            # Compute weights using Gaussian kernel
            weight = gaussian(distances, 2.5)

            # Compute weighted average using matrix multiplication
            numerator = weight @ X
            denominator = weight.sum(1, keepdim=True)

            # Update batch positions
            X[s] = numerator / denominator

    return X

**What does the code above do?**

This is the production-grade mean shift — same logic as `one_update`, but batched, vectorized, and GPU-ready. The structure is:

```
for each iteration (5 total):
    for each batch slice s (n / bs total):
        distances = dist_b(X, X[s])          # (bs, n)
        weights   = gaussian(distances, 2.5)  # (bs, n)
        X[s]      = (weights @ X) / weights.sum(1, keepdim=True)
```

What was a Python loop over 1500 individual points in the slow version is now a Python loop over a small number of batches (e.g. 3, with bs=500). Each batch update does the equivalent of 500 individual point updates in one tensor expression that runs entirely on the GPU's parallel cores.

The `bs` parameter is a memory/speed tradeoff:
- Smaller `bs` → smaller intermediate `(bs, n)` tensor → fits in less GPU memory but fewer points updated per kernel launch.
- Larger `bs` → more GPU parallelism but bigger intermediate tensors.
- On consumer GPUs, `bs` around 500–2000 is usually a good default for n in the low thousands.

---

# Deep Dive: From Sequential to Batched — and the Connection to Attention

The previous deep dive showed that mean shift is gradient ascent on a kernel density estimate. This deep dive is about the *computational* story — how we go from a Python `for` loop over individual points to a single `(weights @ X)` matmul that updates thousands of points in parallel on the GPU. Along the way we'll see that the structure of batched mean shift is *exactly* the structure of an attention mechanism.

---

## The Three Stages of Vectorization

Look at how `one_update` evolves:

### Stage 1 — Pure Python loop (slow)

```python
def one_update(X):
    for i, x in enumerate(X):
        dist = torch.sqrt(((x - X)**2).sum(1))   # (n,)
        weight = tri(dist, 8)                     # (n,)
        X[i] = (weight[:, None] * X).sum(0) / weight.sum()
```

Each iteration of the outer Python loop does $O(n)$ work; we go around $n$ times, so total work is $O(n^2)$. For $n = 1500$ that's 2.25 million elementary ops — manageable, but each Python-loop iteration carries 50× interpreter overhead.

### Stage 2 — Batched (the matmul form)

```python
def meanshift(data, bs=500):
    X = data.clone()
    for it in range(5):
        for i in range(0, n, bs):
            s = slice(i, min(i + bs, n))
            distances = dist_b(X, X[s])             # (bs, n)
            weight = gaussian(distances, 2.5)        # (bs, n)
            X[s] = (weight @ X) / weight.sum(1, keepdim=True)
```

The outer "for i in range(0, n, bs)" loops $n / bs$ times instead of $n$ times. For $n = 1500$, $bs = 500$, that's 3 iterations — a 500× reduction in Python overhead.

The work *inside* the batched loop is $O(bs \cdot n)$ — same total $O(n^2)$ work overall, but now done in vectorized C/CUDA code instead of Python. Two big wins:

- **No Python overhead per point.** The matmul kernel knows about all $bs \cdot n$ work items at once and processes them with optimized SIMD and cache blocking.
- **GPU parallelism.** On a CUDA tensor, the matmul executes on thousands of GPU cores simultaneously. Each thread computes one output element. The same work that would take 7,500 Python-loop iterations on CPU finishes in essentially constant wall-clock time on GPU.

### Stage 3 — The full memory/speed tradeoff

The `bs` parameter is the knob. Smaller batches use less memory but launch more kernels; larger batches use more memory but get more throughput per kernel launch. Tuning `bs` is the same kind of tuning you do for neural network batch sizes — and for the same reasons.

---

## The Shape Trick — Pairwise Distances

The single most consequential line is `dist_b`:

```python
def dist_b(a, b):
    diff = a[None] - b[:, None]          # (1, n, d) - (bs, 1, d) → (bs, n, d)
    return (diff ** 2).sum(2).sqrt()     # → (bs, n)
```

The broadcasting `a[None] - b[:, None]` is the trick that lets us compute all $bs \cdot n$ pairwise differences in *one* tensor expression. It works because:

| Tensor | Before | After reshape | Stride trick |
|---|---|---|---|
| `a` | `(n, d)` | `a[None]: (1, n, d)` | stride along axis 0 is 0 — same memory re-read for every batch |
| `b` | `(bs, d)` | `b[:, None]: (bs, 1, d)` | stride along axis 1 is 0 — same memory re-read for every point |

The subtraction kernel walks the output's `(bs, n, d)` shape, reading `a` and `b` with these zero-stride axes to broadcast. **No memory is allocated for the implicit copies** — broadcasting is just clever stride manipulation on top of a real elementwise kernel.

The same pattern appears everywhere in deep learning: pairwise distances in clustering, query-key similarities in attention, pairwise interactions in graph neural networks. Once you've seen `a[None] - b[:, None]`, you'll recognize it in attention papers, in distance metric learning, in point cloud processing — anywhere two sets of vectors need to be compared element-by-element.

---

## The Weighted Average — As a Matmul

Now look at the second half of the batched update:

```python
weighted_avg = (weight @ X) / weight.sum(1, keepdim=True)
```

`weight` has shape `(bs, n)` and `X` has shape `(n, d)`. The matmul `weight @ X` contracts on the shared `n` axis:

$$(\text{weight @ X})[i, k] \;=\; \sum_{j=1}^{n} \text{weight}[i, j] \cdot X[j, k]$$

That's the weighted sum we want — for every batch query $i$ and every coordinate $k$, sum over all $n$ data points weighted by `weight[i, j]`. Then divide by the row sums to normalize.

Two equivalent ways to write the same thing:

```python
# Form 1: matmul (idiomatic)
weight @ X / weight.sum(1, keepdim=True)

# Form 2: einsum (more explicit about contractions)
torch.einsum('ij,jk->ik', weight, X) / weight.sum(1, keepdim=True)
```

Both compile to the same BLAS call on CPU or cuBLAS call on GPU.

---

## The Attention Connection

Here's the punchline. The full batched mean-shift update is:

```python
distances = dist_b(X, X[s])              # (bs, n)
weight = gaussian(distances, sigma)       # (bs, n) — kernel of distances
X[s] = (weight @ X) / weight.sum(1, keepdim=True)
```

Now look at scaled dot-product attention:

```python
scores  = (Q @ K.transpose(-2, -1)) / sqrt(d)    # (bs, t_q, t_k)
weight  = scores.softmax(dim=-1)                  # (bs, t_q, t_k) — softmax of similarities
output  = weight @ V                              # (bs, t_q, d)
```

The structural parallel:

| Mean Shift | Attention |
|---|---|
| `X[s]` (batch of query points) | `Q` (queries) |
| `X` (all data points) | `K`, `V` (keys, values — typically `K = V` for simple attention) |
| `dist_b(X, X[s])` (negative similarity via distance) | `Q @ K.T` (positive similarity via dot product) |
| `gaussian(distances)` (kernel of distances) | `softmax(scores)` (normalize similarities) |
| `weight @ X` (weighted average of points) | `weight @ V` (weighted average of values) |
| `/ weight.sum(1, keepdim=True)` | (softmax already normalizes) |

**Mean shift is attention where the "values" equal the "keys" and the similarity function is a Gaussian RBF instead of a normalized dot product.**

This isn't a metaphor or a vague analogy — it's a literal structural equivalence. The same broadcasting tricks, the same matmul, the same shape `(bs, n)` weight tensor, the same final `(bs, d)` output. The mathematical *meaning* differs (one is a clustering algorithm; the other is a neural-network primitive), but the *computation* is the same.

This is why the techniques in this notebook scale up directly. Once you know how to vectorize mean shift on the GPU, you know how to vectorize attention on the GPU. The implementations of `MultiHeadAttention` in `torch.nn` use exactly these patterns — with the addition of multiple heads (extra batch axis), learned linear projections for Q/K/V, and softmax instead of Gaussian. The plumbing is identical to what we've already built.

---

## Looking Ahead

When notebook 27 introduces attention, you can stop being surprised by it. The shape gymnastics, the broadcasting, the matmul-as-weighted-average — all of that is already familiar. Attention isn't a new computational primitive; it's mean shift with learnable projections and softmax normalization.

The hard parts of transformers are *what* gets fed in (positional encodings, layer norms, residual connections, feedforward layers) and *what* the projections are trained to do — not the attention computation itself.

### Running on GPU

Now let's move our data to GPU and see the speedup!

**Note**: You need an NVIDIA GPU with CUDA to run this. If you don't have one, it will still work on CPU.

In [ ]:
# ============================================================================
# TRY IT YOURSELF: Interactive Experiments
# ============================================================================
# Uncomment and run these experiments to build intuition!

print("="*70)
print("EXPERIMENT IDEAS - Uncomment the code below to try them!")
print("="*70)

# ═══════════════════════════════════════════════════════════════════════════
# EXPERIMENT 1: Change the bandwidth and see what happens
# ═══════════════════════════════════════════════════════════════════════════
print("""
EXPERIMENT 1: Try different bandwidths
--------------------------------------
Bandwidth controls how far each point "looks" for neighbors.

- Small bandwidth (e.g., 1.0): Only very close points matter
  → Creates MANY small clusters (over-segmentation)
  
- Large bandwidth (e.g., 10.0): Even distant points have influence
  → Creates FEW large clusters (under-segmentation)
  
- Just right (e.g., 2.5): Natural cluster structure emerges

Try modifying the meanshift function to use different bandwidths!
""")

# def meanshift_custom_bw(data, bandwidth=2.5, bs=500):
#     """Same as meanshift but with customizable bandwidth"""
#     n = len(data)
#     X = data.clone()
#     for it in range(5):
#         for i in range(0, n, bs):
#             s = slice(i, min(i + bs, n))
#             distances = dist_b(X, X[s])
#             weight = gaussian(distances, bandwidth)  # ← Custom bandwidth!
#             X[s] = (weight @ X) / weight.sum(1, keepdim=True)
#     return X

# # Try small bandwidth (1.0) - many small clusters
# X_small = meanshift_custom_bw(data, bandwidth=1.0)
# plot_data(centroids+2, X_small, n_samples)
# plt.title("Bandwidth = 1.0 (too small - over-segmentation)")
# plt.show()

# # Try large bandwidth (10.0) - everything merges
# X_large = meanshift_custom_bw(data, bandwidth=10.0)
# plot_data(centroids+2, X_large, n_samples)
# plt.title("Bandwidth = 10.0 (too large - under-segmentation)")
# plt.show()

# ═══════════════════════════════════════════════════════════════════════════
# EXPERIMENT 2: Create different cluster shapes
# ═══════════════════════════════════════════════════════════════════════════
print("""
EXPERIMENT 2: Try different cluster shapes
------------------------------------------
By changing the covariance matrix, you can create:
- Circular clusters: [[5, 0], [0, 5]]
- Wide ellipses: [[20, 0], [0, 2]]
- Tilted ellipses: [[5, 3], [3, 5]]
""")

# # Create elliptical clusters
# ellipse_cov = torch.diag(tensor([20., 2.]))  # Wide in x, narrow in y
# ellipse_dist = MultivariateNormal(tensor([0., 0.]), ellipse_cov)
# ellipse_data = ellipse_dist.sample((200,))
# plt.scatter(ellipse_data[:, 0], ellipse_data[:, 1], s=1)
# plt.title("Elliptical cluster (wide in x, narrow in y)")
# plt.axis('equal')
# plt.show()

# ═══════════════════════════════════════════════════════════════════════════
# EXPERIMENT 3: Watch convergence over iterations
# ═══════════════════════════════════════════════════════════════════════════
print("""
EXPERIMENT 3: Visualize convergence
-----------------------------------
Track how points move over multiple iterations.
You'll see points gradually converge to cluster centers!
""")

# def meanshift_with_history(data, bs=500, n_iter=10):
#     """Returns the clustering state at each iteration"""
#     n = len(data)
#     X = data.clone()
#     history = [X.clone()]  # Save initial state
#     
#     for it in range(n_iter):
#         for i in range(0, n, bs):
#             s = slice(i, min(i + bs, n))
#             distances = dist_b(X, X[s])
#             weight = gaussian(distances, 2.5)
#             X[s] = (weight @ X) / weight.sum(1, keepdim=True)
#         history.append(X.clone())  # Save state after each iteration
#     
#     return history

# # Run and visualize
# history = meanshift_with_history(data)
# fig, axes = plt.subplots(2, 3, figsize=(12, 8))
# for idx, ax in enumerate(axes.flat):
#     if idx < len(history):
#         ax.scatter(history[idx][:, 0], history[idx][:, 1], s=1, alpha=0.5)
#         ax.set_title(f"Iteration {idx}")
# plt.tight_layout()
# plt.show()

print("\nUncomment the code blocks above to run the experiments!")

---
## Common Pitfalls and Debugging Tips

When implementing algorithms like Mean Shift (or any tensor-based code), you'll encounter common issues. Here's how to identify and fix them:

### 1. Shape Mismatch Errors

**The Problem**: PyTorch operations require compatible shapes for broadcasting.

```python
# ❌ WRONG: Forgetting to add dimensions for broadcasting
weight * X  # (1500,) * (1500, 2) → RuntimeError!

# ✅ RIGHT: Add dimension with [:, None] or [None, :]
weight[:, None] * X  # (1500, 1) * (1500, 2) → (1500, 2) ✓
```

**How to debug**: Print shapes at each step!
```python
print(f"weight shape: {weight.shape}")  # Check before operation
print(f"X shape: {X.shape}")
```

### 2. All Points Converge to One Spot

**Symptom**: After running Mean Shift, all points end up at the same location.

**Cause**: Bandwidth is too LARGE - all points influence each other equally, so everything averages to the global center.

**Fix**: Decrease bandwidth (e.g., from 10 to 2.5)

### 3. Too Many Clusters (No Convergence)

**Symptom**: Points don't converge; each point stays roughly where it started.

**Cause**: Bandwidth is too SMALL - only immediate neighbors matter, so there's no "pull" toward cluster centers.

**Fix**: Increase bandwidth

### 4. GPU Out of Memory

**Symptom**: `RuntimeError: CUDA out of memory`

**Cause**: Batch size too large for your GPU memory. The distance matrix has shape (batch_size, n_points).

**Fix**: Reduce `bs` parameter (e.g., from 1000 to 250)

### 5. Very Slow on GPU

**Symptom**: GPU code runs slower than expected (or same as CPU)

**Causes**:
- Data still on CPU (forgot `.cuda()`)
- Moving data between CPU and GPU each iteration (expensive!)

**Fix**: Move data to GPU ONCE at the start
```python
data_gpu = data.cuda()  # Do this once!
result = meanshift(data_gpu)
result_cpu = result.cpu()  # Move back only at the end
```

### Debugging Checklist

```python
# 1. Check tensor shapes
print(f"X shape: {X.shape}")
print(f"weight shape: {weight.shape}")

# 2. Check device (CPU vs GPU)
print(f"X device: {X.device}")  # Should show 'cuda:0' for GPU

# 3. Check for NaN or Inf values (indicates numerical issues)
print(f"Any NaN? {torch.isnan(X).any()}")
print(f"Any Inf? {torch.isinf(X).any()}")

# 4. Check value ranges
print(f"X min: {X.min()}, max: {X.max()}")
print(f"weight min: {weight.min()}, max: {weight.max()}")

# 5. Visualize intermediate results
plt.scatter(X[:, 0].cpu(), X[:, 1].cpu(), s=1)
plt.title("Current state of points")
plt.show()
```

In [ ]:
# ============================================================================
# RUNNING FAST MEAN SHIFT
# ============================================================================

# First run (includes any setup overhead)
X = meanshift(data).cpu()

print("Fast Mean Shift complete!")

**What does the code above do?**

Runs the batched mean shift once on the (possibly GPU-resident) data and moves the result back to CPU for plotting. The first call typically includes some one-time CUDA kernel compilation overhead, so it's slightly slower than steady-state — that's why we time a *second* call in the next cell.

In [ ]:
# ============================================================================
# TIMING THE FAST VERSION
# ============================================================================

# Time multiple runs for accurate measurement
# bs=1250 means we process half the data in each batch (1500/2)
%timeit -n 5 _ = meanshift(data, 1250).cpu()

print()
print("Compare to the slow version: ~450ms")
print("Fast version: ~2ms")
print("Speedup: ~225x faster!")

**What does the code above do?**

Times the batched version. On a modern GPU expect ~2–10 ms per full mean-shift run. Compared to the sequential ~450 ms baseline, that's a 50–200× speedup.

The speedup comes from three sources combined:

1. **Eliminating the Python point-loop** — 1500 iterations of Python overhead replaced by 3 iterations.
2. **Vectorized tensor ops** — `weight @ X` runs in optimized BLAS/cuBLAS C/CUDA code instead of Python.
3. **GPU parallelism** — when running on `.cuda()` tensors, the matmul and elementwise ops execute on thousands of GPU cores at once.

Same algorithm, same results, three orders of magnitude faster wall time. This is the entire performance story of modern deep learning compressed into one notebook.

In [ ]:
# ============================================================================
# VISUALIZING GPU RESULTS
# ============================================================================

# Move back to CPU for plotting
X_cpu = X.cpu() if X.is_cuda else X

plot_data(centroids+2, X_cpu, n_samples)
plt.title("GPU-Accelerated Mean Shift Results")
plt.show()

print("Same results as before, but computed ~200x faster!")

---
## Homework and Further Exploration

### Exercises to Try

1. **Implement K-Means clustering** on GPU
   - Pick k random centroids
   - Assign each point to nearest centroid
   - Update centroids as mean of assigned points
   - Repeat until convergence

2. **Implement DBSCAN** (Density-Based Spatial Clustering)
   - Groups points that are closely packed together
   - Can find clusters of arbitrary shape
   - Identifies noise points

3. **Locality Sensitive Hashing**
   - Approximate nearest neighbor search
   - Essential for scaling to large datasets

### Bonus Challenges

**Super bonus**: Invent a new Mean Shift algorithm which picks only the closest points, avoiding the quadratic time complexity!

**Super super bonus**: Publish a paper describing your improvement! 🎓

### Connection to Diffusion Models

As you learn more about Stable Diffusion, look for these connections:
- **Latent space clustering**: How do generated images organize in latent space?
- **Attention as weighted averaging**: Self-attention uses similar weighted averaging concepts
- **Denoising as movement**: Points moving toward high-density regions is like denoising!

---

# Summary: What We Built and Why It Matters

We started with 1,500 unlabeled 2-D points and ended with an algorithm that automatically discovers 6 clusters in milliseconds on a GPU. Along the way we touched four ideas that will recur for the rest of the course.

## What we covered

| Section | Key idea |
|---|---|
| §2 Data | Multivariate-normal sampling builds clustered ground truth. Covariance matrices control cluster shape. |
| §3 Algorithm | Mean shift = "move each point toward the weighted mean of its neighbors." Iterate to convergence. |
| §4 Kernels | Gaussian and triangular kernels both work; bandwidth is the single hyperparameter that matters. |
| §5 Sequential | A Python loop implementation that is correct but slow — $O(n^2)$ work compounded by Python overhead. |
| §6 Convergence | Mean shift is gradient ascent on a kernel density estimate; that's why it converges to density modes. |
| §7 Batched | Replacing the point-loop with a `(weight @ X)` matmul gives ~100× speedup before we even reach the GPU. |
| §8 Attention | The batched update is structurally identical to softmax attention — preview of notebook 27. |
| §9 GPU | One `.cuda()` call buys another ~10–100× via parallel hardware. |

## The big takeaways

1. **Mean shift is gradient ascent in disguise.** Once you see the math (§6 deep dive), you understand *why* it finds density peaks. No magic, no heuristics — just calculus on the kernel density estimate.

2. **Pairwise operations are broadcasting + matmul.** Whether you're computing distances (mean shift), similarities (attention), or interactions (graph nets), the shape pattern `a[None] - b[:, None]` followed by some reduction is the workhorse. Internalize it once and you'll see it everywhere.

3. **Batching is the entry ticket to the GPU.** A Python-loop algorithm and a batched-tensor algorithm do the same arithmetic but pay vastly different wall-clock costs. The gap only grows on bigger problems. Vectorize before you optimize anything else.

4. **The line between "clustering" and "attention" is thinner than it looks.** §8's deep dive showed that the batched mean-shift update is the same operation as softmax attention with the values equal to the keys and the similarity being a Gaussian RBF. When notebook 27 introduces attention, it will look familiar.

## Suggested next steps

1. **Read through the three deep dives** (kernels & bandwidth, convergence, batched + attention). They're the conceptual content that pays off for the rest of the course.
2. **Try the experiments in the source notebook** — different bandwidths, different cluster shapes, history-recording mean shift to visualize convergence step by step.
3. **Run `concept-extraction` on this notebook** to produce cards: "what is a kernel," "why is mean shift gradient ascent," "what is the pairwise distance broadcasting pattern," "what is the attention connection."
4. **Notebook 03** (backprop) is next — it uses the same matmul vectorization machinery on a different problem (computing gradients through a small neural network).